# Inconsistency Masks: Study A Reproduction Script

**Paper:** *Inconsistency Masks: Harnessing Model Disagreement for Stable Semi-Supervised Segmentation* **Codebase:** Study A (Pre-trained Benchmarks on Cityscapes)

This notebook implements the **Generational Inconsistency Mask (IM)** framework. It is designed to reproduce the exact results presented in **Table 1** of the research paper.

---

### 📋 Prerequisites

* **Python 3.8+** & **PyTorch 2.0+**
* **Libraries:** `segmentation-models-pytorch`, `timm`, `scipy`, `pandas`
* **Dataset:** Cityscapes
    * **Option A (Automatic):** Place `leftImg8bit_trainvaltest.zip` and `gtFine_trainvaltest.zip` in your data directory. The script will automatically unzip them.
    * **Option B (Manual):** Extract the dataset manually and set `DATASET_ROOT` in the code below.

---

### ⚙️ Experiment Configuration

Control the experiment using the variables in **Section 1 (Setup & Configuration)**.

#### 1. Method Selection (`SELECTED_METHOD`)
Change the `SELECTED_METHOD` variable to switch between the paper's baselines:

| Variable Value | Paper Method (Table 1) | Architecture | Note |
| :--- | :--- | :--- | :--- |
| **`"IMAS"`** | iMAS + IM | ResNet-50 | Adaptive Supervision |
| **`"U2PL"`** | U2PL + IM | ResNet-50 | Contrastive Learning |
| **`"UNIMATCH"`** | UniMatch + IM | ResNet-50 | Dual-stream consistency |
| **`"UNIMATCH_V2"`** | UniMatch v2 + IM | DINOv2-Small | Foundation Model |
| **`"SEGKC"`** | SegKC + IM | DINOv2-Small & Base | Knowledge Distillation |

#### 2. Data Split (`SELECTED_FRACTION`)
* **Options:** `"1_2"`, `"1_4"`, `"1_8"`, `"1_16"`, `"1_30"`
* *Note:* Matches the standard Cityscapes semi-supervised partitions used in the publication.

---

### 🚀 Usage Instructions
1.  **Configure:** Edit `SELECTED_METHOD` and `SELECTED_FRACTION` in the code cell below.
2.  **Run All:** Execute all cells (`Runtime` > `Run all`).
3.  **Outputs:**
    * **Gen 0:** Base Supervised or SSL method training.
    * **Gen 1+:** Inconsistency Mask generation & Self-Training.
    * **Logs:** Saved to `training_log.csv`.

In [ ]:
# ==============================================================================
# --- 1) SETUP & CONFIGURATION ---
# ==============================================================================
import os
import sys
import torch

# --- 1. Environment Detection ---
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Detected Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("✅ Detected Local environment.")

# --- 2. User Configuration ---

# [!] CHANGE THIS to switch methods: "IMAS", "U2PL", "UNIMATCH", "UNIMATCH_V2", "SEGKC"
SELECTED_METHOD = "U2PL"

# [!] DATA SPLIT: "1_2", "1_4", "1_8", "1_16"
SELECTED_FRACTION = "1_16"

# [!] DATASET PATHS
if IN_COLAB:
    # Colab Paths
    DATASET_ROOT = "/content/cityscapes"
    DRIVE_EXP_DIR = "/content/drive/My Drive/Cityscapes_Generational_IM"

    # Mount Drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    # Install dependencies
    !pip install -q segmentation-models-pytorch
else:
    # Local Paths (Edit these)
    DATASET_ROOT = "./data/cityscapes"
    DRIVE_EXP_DIR = "./experiments"

# Ensure directories exist
os.makedirs(DRIVE_EXP_DIR, exist_ok=True)
print(f"📂 Dataset Root: {DATASET_ROOT}")
print(f"📂 Experiment Output: {DRIVE_EXP_DIR}")


# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch import amp
from torch.utils.checkpoint import checkpoint
import torchvision.transforms.v2 as v2
import torchvision.transforms.v2.functional as Fv2
from torchvision.transforms import InterpolationMode
import scipy.stats as stats
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.decoders.deeplabv3.decoder import DeepLabV3PlusDecoder, ASPP
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.notebook import tqdm
import operator
import gc
import numpy as np
import json
import random
import time
import csv
import inspect
import math
import copy
from datetime import datetime, timezone
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="torch.optim.lr_scheduler")

# --- ⚙️ GLOBAL CONFIGURATION ---
EXPERIMENT_NAME = f"Cityscapes_{SELECTED_METHOD}_{SELECTED_FRACTION}_test"

# 1. Base Defaults (Will be overridden by Method Profiles)
HPARAMS = {

    # [!] Controlled by User Config above
    "DATASET_FRACTION": SELECTED_FRACTION,
    "SSL_METHOD": SELECTED_METHOD,

    "NUM_GENERATIONS": 10,
    "USE_CPS_DATA_SPLIT": True,
    "SEED": 806,
    "BACKBONE": "resnet50",
    "NUM_CLASSES": 19,
    "IGNORE_INDEX": 255,

    # Validation Control
    # Modes: "standard" (Fast/Resize) or "sliding" (Accurate/Slow)
    "VALIDATION_MODE": "sliding",

    # Training Tricks
    "USE_RCS": False,
    "SKIP_TEACHER_VAL_SUP": False,

    "DROPOUT_SCHEDULE": {
        "GEN0": 0.0,
        "GENX": 0.0
    },

    # Gen 1+ Augmentations
    "SUP_AUG": { "CUTMIX_PROB": 0.0, "CLASSMIX_PROB": 0.5 },

    # Optimizer Configs (Definitions only, selected by Profile)
    "OPTIMIZERS": {
        "SGD":      {"LR_HEAD": 0.01, "LR_BACKBONE": 0.01, "WEIGHT_DECAY": 5e-4, "MOMENTUM": 0.9, "NESTEROV": False},
        "ADAMW":    {"LR_HEAD": 1e-3, "LR_BACKBONE": 1e-4, "WEIGHT_DECAY": 2e-2, "BETAS": (0.9, 0.999)},
        "ADAMW_V2": {"LR_HEAD": 2e-4, "LR_BACKBONE": 5e-6, "WEIGHT_DECAY": 1e-2, "BETAS": (0.9, 0.999)},
    },

    "LR_FINETUNE_FACTOR": 0.1,

    # Epochs & Snapshots
    "STUDENTS_GEN0": 1,
    "NUM_EPOCHS_GEN_0": 150,
    "WARMUP_PCT_GEN0": 0.00,
    "NUM_EPOCHS_GEN_X": 30,
    "VAL_START_EPOCH_GEN0": 1,
    "VAL_START_EPOCH_GENX": 1,
    "SNAPSHOT_MODELS_GEN0": 5,
    "SNAPSHOT_MODELS_GENX": 5,
    "SNAPSHOT_WARMUP_PCT": 1.0, # no snapshot
    "EMA_ALPHA_GENX": 0.99,

    # Inconsistency Masks
    "IM_TEACHERS_TOPK": 2,
    "IM_ERODE": 3,
    "IM_DILATE": 5,
    "IM_TTA": True,
    "SW_OVERLAP": 2/3,

    # Augmentations
    "SCALE_MIN": 0.5, "SCALE_MAX": 2.0,
    "USE_STRONG_PSEUDO_AUG": True,

    "AUG_SCHEDULE_GEN0": {
        "ENABLED": True, "MODE": "linear", "SCOPE": "step",
        "START": {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.30,"cj_hue":0.10,"cj_p":0.80,"blur_p":0.50,"blur_sigma_max":1.50,"gray_p":0.00},
        "END":   {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.55,"cj_hue":0.20,"cj_p":0.95,"blur_p":0.75,"blur_sigma_max":1.50,"gray_p":0.15}
    },
    "AUG_SCHEDULE_GENX": {
        "ENABLED": True, "MODE": "linear", "SCOPE": "step",
        "START": {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.30,"cj_hue":0.10,"cj_p":0.80,"blur_p":0.50,"blur_sigma_max":1.50,"gray_p":0.00},
        "END":   {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.60,"cj_hue":0.20,"cj_p":0.95,"blur_p":0.80,"blur_sigma_max":1.50,"gray_p":0.20}
    },
    "AUG_SCHEDULE_GENX_PSEUDO": {
        "ENABLED": True, "MODE": "linear", "SCOPE": "step",
        "START": {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.6,"cj_hue":0.2,"cj_p":0.90,"blur_p":0.60,"blur_sigma_max":1.50,"gray_p":0.05,"strong_ops_p":2.0},
        "END":   {"scale_min":0.5,"scale_max":2.0,"cj_mag":0.75,"cj_hue":0.25,"cj_p":0.98,"blur_p":0.90,"blur_sigma_max":1.80,"gray_p":0.30,"strong_ops_p":4.0}
    },

    # Losses
    "LOSS_GEN0": "ohem",
    "LOSS_GENX": "ce_lovasz",
    "OHEM_KEEP": 0.25, "OHEM_MIN_KEPT": 100000, "OHEM_THRESH": 0.7,

    # =========================================================
    # --- METHOD PROFILES (The "Auto-Config" Logic) ---
    # =========================================================
    "ALGORITHM_CONFIGS": {

        "SUPERVISED": {
            "BATCH_SIZE_GEN0": 16,
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 768,
            "ACTIVE_OPTIMIZER": "ADAMW",
            "USE_RCS": False,
            "LOSS_GEN0": "ce_lovasz",
            "CUTMIX_PROB": 0.0,    # Box-based mixing
            "CLASSMIX_PROB": 0.0,  # Class-boundary mixing
        },

        "IMAS": {
            "BATCH_SIZE_GEN0": 16,
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 768,
            "ACTIVE_OPTIMIZER": "ADAMW",
            "USE_RCS": False,
            # IMAS Specifics
            "EMA_ALPHA": 0.996, "LAMBDA_U": 3.0, "CONF_TAU": 0.7, "USE_CUTMIX": True,
            "LOSS_GEN0": "ohem",
        },

        "SEGKC": {
            "BATCH_SIZE_GEN0": 6,     # Reduced (Training 2 models: Base + Small)
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 798,
            "ACTIVE_OPTIMIZER": "ADAMW_V2",
            "USE_RCS": False,
            "NUM_EPOCHS_GEN_0": 200,

            # SegKC Specifics
            "SENIOR_BACKBONE": "dinov2_vitb14", # The "Teacher" (Trainable)
            "JUNIOR_BACKBONE": "dinov2_vits14", # The "Student" (Target)
            "CONF_THRESH": 0.95,

            # Loss Weights (Eq 10)
            "LAMBDA_SUP": 1.0,
            "LAMBDA_CON": 1.0,
            "LAMBDA_KD": 10.0,      # Distillation weight (usually high)
            "TEMP": 1.0,            # Temperature for KD

            "LOSS_GEN0": "ce",
        },

        "UNIMATCH": {
            "BATCH_SIZE_GEN0": 16,
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 768,
            "ACTIVE_OPTIMIZER": "ADAMW",
            "USE_RCS": False,
            # UniMatch Specifics
            "EMA_ALPHA": 0.999, "CONF_THRESH": 0.0, "LAMBDA_U": 1.0, "FP_DROP_RATE": 0.5, "USE_CUTMIX": True,
            "LOSS_GEN0": "ohem",
        },

        "UNIMATCH_V2": {
            "BATCH_SIZE_GEN0": 8,
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 798, # Divisible by 14
            "ACTIVE_OPTIMIZER": "ADAMW_V2",
            "USE_RCS": False,
            "NUM_EPOCHS_GEN_0": 150,
            "VALIDATION_MODE": "sliding",
            # V2 Specifics
            "EMA_ALPHA": 0.996,
            "CONF_THRESH": 0.0,
            "LAMBDA_U": 1.0,
            "USE_CUTMIX": True,
            "LOSS_GEN0": "ohem",
        },

        "U2PL": {
            "BATCH_SIZE_GEN0": 8,
            "BATCH_SIZE": 16,
            "VAL_BATCH_SIZE": 16,
            "CROP_SIZE": 768,
            "ACTIVE_OPTIMIZER": "ADAMW",
            "USE_RCS": False,
            # U2PL Specifics
            "EMA_ALPHA": 0.999, "LAMBDA_U": 1.0, "LAMBDA_C": 0.1, "DROP_PERCENT": 30.0,
            "USE_CUTMIX": True, "QUEUE_LEN": 30000, "TEMP": 0.5,
            "LOSS_GEN0": "ohem",
        }
    }
}

# --- AUTO-APPLY PROFILE ---
selected_method = HPARAMS["SSL_METHOD"]
if selected_method in HPARAMS["ALGORITHM_CONFIGS"]:
    profile = HPARAMS["ALGORITHM_CONFIGS"][selected_method]
    print(f"⚙️ Applying Auto-Config for {selected_method}:")
    for key, value in profile.items():
        if key != "ALGORITHM_CONFIGS":
            HPARAMS[key] = value
            print(f"   -> Set {key} = {value}")

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False

seed_all(HPARAMS["SEED"])
torch.backends.cudnn.benchmark = True
print(f"✅ Configuration '{EXPERIMENT_NAME}' loaded with method: {HPARAMS['SSL_METHOD']}")

In [ ]:
# ==============================================================================
# --- 2) DATASET PREPARATION & SPLITTING ---
# ==============================================================================

# --- A. Unzip Dataset ---
drive_folder_name = "Cityscapes_Dataset" # Check this matches your Drive folder
drive_base_path = os.path.join("/content/drive/My Drive", drive_folder_name)
images_zip_path = os.path.join(drive_base_path, "leftImg8bit_trainvaltest.zip")
labels_zip_path = os.path.join(drive_base_path, "gtFine_trainvaltest.zip")
DATASET_ROOT    = "/content/cityscapes"

if not os.path.isdir(os.path.join(DATASET_ROOT, "leftImg8bit")):
    print(f"⏳ Unzipping dataset to {DATASET_ROOT}...")
    if os.path.exists(images_zip_path) and os.path.exists(labels_zip_path):
        !unzip -oq "{images_zip_path}" -d "{DATASET_ROOT}"
        !unzip -oq "{labels_zip_path}" -d "{DATASET_ROOT}"
        print("✅ Dataset unzipped.")
    else:
        raise FileNotFoundError(f"❌ Zips not found in {drive_base_path}")
else:
    print("✅ Dataset already unzipped.")

# --- B. Create Data Splits ---
split_dir = os.path.join(DATASET_ROOT, "splits")
os.makedirs(split_dir, exist_ok=True)

labeled_split_file   = os.path.join(split_dir, f"labeled_{HPARAMS['DATASET_FRACTION']}.txt")
unlabeled_split_file = os.path.join(split_dir, f"unlabeled_{HPARAMS['DATASET_FRACTION']}.txt")
val_split_file       = os.path.join(split_dir, "val.txt")

# 1. Training Splits
if HPARAMS["USE_CPS_DATA_SPLIT"]:
    # Copy from Drive (Author's splits)
    print(f"➡️ Using pre-defined iMAS splits for {HPARAMS['DATASET_FRACTION']}")
    imas_split_dir = os.path.join(drive_base_path, HPARAMS['DATASET_FRACTION'])

    def process_split(src_name, dest_path):
        src = os.path.join(imas_split_dir, src_name)
        if not os.path.exists(src): raise FileNotFoundError(f"Missing {src}")
        with open(src, 'r') as f_in, open(dest_path, 'w') as f_out:
            lines = f_in.read().splitlines()
            # Add absolute path prefix
            for line in lines:
                if line.strip(): f_out.write(os.path.join(DATASET_ROOT, line.strip()) + "\n")
        return len(lines)

    n_lab = process_split("labeled.txt", labeled_split_file)
    n_ulb = process_split("unlabeled.txt", unlabeled_split_file)
    print(f"   -> Loaded {n_lab} labeled, {n_ulb} unlabeled.")

else:
    # Generate Random Splits
    print(f"➡️ Generating random splits (Seed: {HPARAMS['SEED']})")
    train_dir = os.path.join(DATASET_ROOT, "leftImg8bit/train")
    all_images = []
    for root, _, files in os.walk(train_dir):
        for name in files:
            if name.endswith(".png"): all_images.append(os.path.join(root, name))
    all_images.sort()

    # Parse fraction string "1_16" -> 1/16
    num, den = map(int, HPARAMS['DATASET_FRACTION'].split('_'))
    fraction = num / den

    # Shuffle & Split
    np.random.seed(HPARAMS['SEED'])
    np.random.shuffle(all_images)

    n_total = len(all_images)
    n_labeled = int(n_total * fraction)

    list_lab = all_images[:n_labeled]
    list_ulb = all_images[n_labeled:]

    with open(labeled_split_file, 'w') as f: f.write("\n".join(list_lab))
    with open(unlabeled_split_file, 'w') as f: f.write("\n".join(list_ulb))
    print(f"   -> Generated {len(list_lab)} labeled, {len(list_ulb)} unlabeled.")

# 2. Validation Split (Fixed)
if not os.path.exists(val_split_file):
    val_dir = os.path.join(DATASET_ROOT, "leftImg8bit/val")
    val_images = []
    for root, _, files in os.walk(val_dir):
        for name in files:
            if name.endswith(".png"): val_images.append(os.path.join(root, name))
    val_images.sort()
    with open(val_split_file, 'w') as f: f.write("\n".join(val_images))
    print(f"✅ Validation split created ({len(val_images)} images).")
else:
    print(f"✅ Validation split exists.")

# --- Dirs for Outputs ---
DRIVE_EXP_DIR = os.path.join("/content/drive/My Drive/Cityscapes_Generational_IM", EXPERIMENT_NAME)
MODEL_SAVE_DIR = os.path.join(DRIVE_EXP_DIR, "models")
TRAIN_LOG_PATH = os.path.join(DRIVE_EXP_DIR, "training_log.csv")
BEST_LOG_PATH  = os.path.join(DRIVE_EXP_DIR, "best_miou_records.csv")
PSEUDO_LABEL_DIR = os.path.join(DRIVE_EXP_DIR, "pseudo_labels")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(PSEUDO_LABEL_DIR, exist_ok=True)

# --- Constants for Classes ---
# Cityscapes ID -> TrainID
ID_TO_TRAINID = np.array([HPARAMS["IGNORE_INDEX"]] * 256, dtype=np.uint8)
_map = {7:0, 8:1, 11:2, 12:3, 13:4, 17:5, 19:6, 20:7, 21:8, 22:9, 23:10, 24:11, 25:12, 26:13, 27:14, 28:15, 31:16, 32:17, 33:18}
for k, v in _map.items(): ID_TO_TRAINID[k] = v

CITYSCAPES_CLASS_NAMES = [
    "road","sidewalk","building","wall","fence","pole",
    "traffic light","traffic sign","vegetation","terrain","sky",
    "person","rider","car","truck","bus","train","motorcycle","bicycle"
]

In [ ]:
# ==============================================================================
# --- 3) DATA PIPELINE & MODEL ARCHITECTURE ---
# ==============================================================================

# --- A. Augmentation Schedule Controller ---
class AugScheduleController:
    """Handles dynamic augmentation parameters (weak/strong) over training steps."""
    def __init__(self, cfg, normalize_mean=[0.485,0.456,0.406], normalize_std=[0.229,0.224,0.225]):
        self.enabled = bool(cfg.get("ENABLED", False))
        self.start   = cfg.get("START", {})
        self.end     = cfg.get("END", {})
        self.total_steps = 1
        self.current_step = 0
        self.normalize = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True),
                                     v2.Normalize(normalize_mean, normalize_std)])

    def set_total(self, total_steps):
        self.total_steps = max(1, int(total_steps))
        self.current_step = min(self.current_step, self.total_steps)

    def advance(self, n=1):
        self.current_step = min(self.total_steps, self.current_step + int(n))

    def _progress(self):
        if not self.enabled: return 0.0
        return float(self.current_step) / float(self.total_steps)

    def _param(self, key, default):
        a = self.start.get(key, default)
        b = self.end.get(key, a)
        t = self._progress()
        return a + (b - a) * t

    def get_scale_range(self):
        if not self.enabled: return HPARAMS["SCALE_MIN"], HPARAMS["SCALE_MAX"]
        return self._param("scale_min", 0.5), self._param("scale_max", 2.0)

    def get_transforms(self, strong: bool = False):
        # Interpolate params
        cj_mag = self._param("cj_mag", 0.3)
        cj_hue = self._param("cj_hue", 0.1)
        cj_p   = self._param("cj_p", 0.8)
        blur_p = self._param("blur_p", 0.5)
        sig_mx = self._param("blur_sigma_max", 1.5)
        gray_p = self._param("gray_p", 0.0)

        ops = [v2.ToImage()]

        if strong:
            # Strong Aug Logic (RandAugment-like)
            strong_p = self._param("strong_ops_p", 0.0)
            k_strong = max(0, min(4, int(round(strong_p))))

            class RandomizedPosterize(nn.Module):
                def forward(self, img): return Fv2.posterize(img, bits=random.choice([4,5,6,7]))
            class RandomizedSolarize(nn.Module):
                def forward(self, img): return Fv2.solarize(img, threshold=random.uniform(0.25, 0.75))
            class RandomizedSharpness(nn.Module):
                def forward(self, img): return Fv2.adjust_sharpness(img, sharpness_factor=random.uniform(1.5, 2.5))

            uint8_ops = [RandomizedPosterize()]
            float_ops = [
                v2.ColorJitter(cj_mag, cj_mag, cj_mag, cj_hue),
                v2.GaussianBlur(3, (0.1, float(sig_mx))),
                v2.RandomInvert(), v2.RandomAutocontrast(), v2.RandomEqualize(),
                RandomizedSolarize(), RandomizedSharpness()
            ]

            if k_strong > 0:
                total_ops = len(uint8_ops) + len(float_ops)
                idxs = torch.randperm(total_ops)[:k_strong].tolist()
                U = len(uint8_ops)
                for i in idxs:
                    if i < U: ops.append(uint8_ops[i])
                ops.append(v2.ToDtype(torch.float32, scale=True))
                for i in idxs:
                    if i >= U: ops.append(float_ops[i-U])
            else:
                ops.append(v2.ToDtype(torch.float32, scale=True))
        else:
            # Weak Augs
            ops.append(v2.ToDtype(torch.float32, scale=True))
            ops.append(v2.RandomApply([v2.ColorJitter(cj_mag, cj_mag, cj_mag, cj_hue)], p=float(cj_p)))
            ops.append(v2.RandomGrayscale(p=float(gray_p)))
            ops.append(v2.RandomApply([v2.GaussianBlur(3, (0.1, float(sig_mx)))], p=float(blur_p)))

        ops.append(v2.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]))
        return v2.Compose(ops)


# --- B. Samplers & Loaders ---
class RCSSampler(torch.utils.data.Sampler):
    """Rare Class Sampling: Oversamples images containing rare classes."""
    def __init__(self, dataset, id_to_trainid, num_samples=None):
        self.dataset = dataset
        self.indices = list(range(len(dataset)))
        self.num_samples = len(dataset) if num_samples is None else num_samples
        # Rare TrainIDs: Wall(3), Fence(4), Pole(5), TL(6), TS(7), Rider(12), Bus(15), Train(16), MC(17), Bike(18)
        self.rare_classes = [3, 4, 5, 6, 7, 12, 15, 16, 17, 18]
        self.rare_indices = []

        print(f"⚖️  [RCS] Scanning {len(dataset)} images for rare classes...")
        for idx in self.indices:
            try:
                # Access path directly from dataset's internal list
                img_path = dataset.image_paths[idx]
                img_dir, img_name = os.path.split(img_path)
                mask_dir = img_dir.replace('leftImg8bit', 'gtFine')
                mask_name = img_name.replace('_leftImg8bit.png', '_gtFine_labelIds.png')
                mask_path = os.path.join(mask_dir, mask_name)

                if os.path.exists(mask_path):
                    mask = np.array(Image.open(mask_path))
                    mask_trainid = id_to_trainid[mask]
                    if np.any(np.isin(mask_trainid, self.rare_classes)):
                        self.rare_indices.append(idx)
            except: pass
        print(f"   -> Found {len(self.rare_indices)} rare samples.")

    def __iter__(self):
        n = self.num_samples
        out = []
        if not self.rare_indices: return iter(np.random.choice(self.indices, n, replace=True))
        for _ in range(n):
            if random.random() < 0.5: out.append(random.choice(self.rare_indices))
            else: out.append(random.choice(self.indices))
        return iter(out)

    def __len__(self): return self.num_samples


# --- C. Model Architecture (ResNet50d + DeepLabV3+) ---

def download_backbone_weights(dest_path):
    """Downloads ResNet50d weights if not present."""
    if os.path.exists(dest_path): return
    url = "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/resnet50d_ra2-464e36ba.pth"
    print(f"⬇️  Downloading ResNet50d weights to {dest_path}...")
    !wget -q -O "{dest_path}" "{url}"

# 1. SyncBN & Helper Modules
def get_syncbn(): return nn.BatchNorm2d

class ASPP(nn.Module):
    def __init__(self, in_planes, inner_planes=256, sync_bn=False, dilations=(12, 24, 36)):
        super(ASPP, self).__init__()
        norm_layer = get_syncbn() if sync_bn else nn.BatchNorm2d
        self.conv1 = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)),
            nn.Conv2d(in_planes, inner_planes, kernel_size=1, bias=False), norm_layer(inner_planes), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(nn.Conv2d(in_planes, inner_planes, kernel_size=1, bias=False), norm_layer(inner_planes), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(nn.Conv2d(in_planes, inner_planes, kernel_size=3, padding=dilations[0], dilation=dilations[0], bias=False), norm_layer(inner_planes), nn.ReLU(inplace=True))
        self.conv4 = nn.Sequential(nn.Conv2d(in_planes, inner_planes, kernel_size=3, padding=dilations[1], dilation=dilations[1], bias=False), norm_layer(inner_planes), nn.ReLU(inplace=True))
        self.conv5 = nn.Sequential(nn.Conv2d(in_planes, inner_planes, kernel_size=3, padding=dilations[2], dilation=dilations[2], bias=False), norm_layer(inner_planes), nn.ReLU(inplace=True))
        self.out_planes = (len(dilations) + 2) * inner_planes

    def get_outplanes(self): return self.out_planes
    def forward(self, x):
        _, _, h, w = x.size()
        feat1 = F.interpolate(self.conv1(x), size=(h, w), mode="bilinear", align_corners=True)
        return torch.cat((feat1, self.conv2(x), self.conv3(x), self.conv4(x), self.conv5(x)), 1)

# 2. ResNet Encoder
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=dilation, groups=groups, bias=False, dilation=dilation)
def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class Bottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1, norm_layer=nn.BatchNorm2d):
        super(Bottleneck, self).__init__()
        width = int(planes * (base_width / 64.0)) * groups
        self.conv1 = conv1x1(inplanes, width)
        self.bn1 = norm_layer(width)
        self.conv2 = conv3x3(width, width, stride, groups, dilation)
        self.bn2 = norm_layer(width)
        self.conv3 = conv1x1(width, planes * self.expansion)
        self.bn3 = norm_layer(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward_impl(self, x):
        # Separation of logic for checkpointing
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out

    def forward(self, x):
        # Use Checkpointing only if gradients are required (Training)
        if self.training and x.requires_grad:
            out = checkpoint(self.forward_impl, x, use_reentrant=False)
        else:
            out = self.forward_impl(x)
        return self.relu(out)

class ResNet(nn.Module):
    def __init__(self, block, layers, replace_stride_with_dilation=[False, False, False], multi_grid=False, fpn=False):
        super(ResNet, self).__init__()
        norm_layer = nn.BatchNorm2d
        self._norm_layer = norm_layer
        self.inplanes = 64
        self.dilation = 1
        self.groups = 1
        self.base_width = 64
        self.fpn = fpn
        self.conv1 = nn.Sequential(
            conv3x3(3, 32, stride=2), norm_layer(32), nn.ReLU(inplace=True),
            conv3x3(32, 32), norm_layer(32), nn.ReLU(inplace=True),
            conv3x3(32, 64))
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=True)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2], multi_grid=multi_grid)
        for m in self.modules():
            if isinstance(m, Bottleneck): nn.init.constant_(m.bn3.weight, 0)
    def get_outplanes(self): return self.inplanes
    def _make_layer(self, block, planes, blocks, stride=1, dilate=False, multi_grid=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(conv1x1(self.inplanes, planes * block.expansion, stride), norm_layer(planes * block.expansion))
        grids = [2, 2, 4] if multi_grid else [1] * blocks
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups, self.base_width, previous_dilation * grids[0], norm_layer))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, 1, None, self.groups, self.base_width, self.dilation * grids[i], norm_layer))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return [x1, x2, x3, x4] if self.fpn else [x3, x4]

# 3. Decoder & Full Model
class dec_deeplabv3_plus(nn.Module):
    def __init__(self, in_planes, num_classes=19, inner_planes=256, dilations=(12, 24, 36), low_conv_planes=48, dropout_p=0.0):
        super(dec_deeplabv3_plus, self).__init__()
        norm_layer = nn.BatchNorm2d
        self.low_conv = nn.Sequential(nn.Conv2d(256, low_conv_planes, kernel_size=1), norm_layer(low_conv_planes), nn.ReLU(inplace=True))
        self.aspp = ASPP(in_planes, inner_planes=inner_planes, dilations=dilations)
        self.head = nn.Sequential(nn.Conv2d(self.aspp.get_outplanes(), 256, 1, bias=False), norm_layer(256), nn.ReLU(inplace=True))

        # Split classifier to access features
        self.classifier_conv = nn.Sequential(
            nn.Conv2d(256+int(low_conv_planes), 256, 3, 1, 1, bias=False), norm_layer(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, 1, 1, bias=False), norm_layer(256), nn.ReLU(inplace=True)
        )

        self.dropout = nn.Dropout2d(p=dropout_p)

        self.classifier_final = nn.Conv2d(256, num_classes, 1, 1, 0)

    def forward(self, x, return_features=False):
        x1, x2, x3, x4 = x
        aspp_out = self.head(self.aspp(x4))
        low_feat = self.low_conv(x1)
        h, w = low_feat.size()[-2:]
        aspp_out = F.interpolate(aspp_out, size=(h, w), mode="bilinear", align_corners=True)

        concat_feats = torch.cat((low_feat, aspp_out), dim=1)
        features = self.classifier_conv(concat_feats)

        features_drop = self.dropout(features)
        logits = self.classifier_final(features)

        if return_features:
            return logits, features
        return logits

class Aux_Module(nn.Module):
    def __init__(self, in_planes, num_classes=19):
        super(Aux_Module, self).__init__()
        self.aux = nn.Sequential(nn.Conv2d(in_planes, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True), nn.Conv2d(256, num_classes, 1, 1, 0))
    def forward(self, x): return self.aux(x)

class iMAS_Model(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super(iMAS_Model, self).__init__()
        self.encoder = ResNet(Bottleneck, [3, 4, 6, 3], replace_stride_with_dilation=[False, False, True], multi_grid=True, fpn=True)
        self.decoder = dec_deeplabv3_plus(in_planes=self.encoder.get_outplanes(), num_classes=num_classes, low_conv_planes=48, dropout_p=dropout_p)
        self.auxor = Aux_Module(1024, num_classes)
    def forward(self, x):
        feats = self.encoder(x)
        main_out = self.decoder(feats)
        h, w = x.shape[-2:]
        main_out = F.interpolate(main_out, (h, w), mode="bilinear", align_corners=True)
        if self.training:
            aux_out = F.interpolate(self.auxor(feats[2]), (h, w), mode="bilinear", align_corners=True)
            return main_out, aux_out
        return main_out

# --- U2PL Specific Model (Faithful Reproduction) ---
class U2PL_Model(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super(U2PL_Model, self).__init__()
        self.encoder = ResNet(Bottleneck, [3, 4, 6, 3], replace_stride_with_dilation=[False, False, True], multi_grid=True, fpn=True)
        self.decoder = dec_deeplabv3_plus(in_planes=self.encoder.get_outplanes(), num_classes=num_classes, low_conv_planes=48, dropout_p=dropout_p)
        self.auxor = Aux_Module(1024, num_classes)

        # U2PL Projection Head
        self.rep_head = nn.Sequential(
            nn.Conv2d(256, 256, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 1, stride=1, padding=0, bias=True)
        )

    def forward(self, x, return_features=False):
        # Encoder
        feats = self.encoder(x)

        # Decoder Logic
        # We only calculate deep features if we strictly need them for U2PL training
        if return_features:
            main_out, dec_feats = self.decoder(feats, return_features=True)
        else:
            main_out = self.decoder(feats, return_features=False)
            dec_feats = None

        h, w = x.shape[-2:]
        main_out = F.interpolate(main_out, (h, w), mode="bilinear", align_corners=True)

        # Aux Head (Always needed during training)
        if self.training:
            aux_out = F.interpolate(self.auxor(feats[2]), (h, w), mode="bilinear", align_corners=True)
        else:
            aux_out = None

        # --- Return Logic ---

        # Mode 1: U2PL Training (Explicitly requested features)
        # Returns: (Logits, Aux, Reps)
        if return_features:
            rep_out = self.rep_head(dec_feats)
            rep_out = F.normalize(rep_out, dim=1)
            rep_out = F.interpolate(rep_out, (h, w), mode="bilinear", align_corners=True)
            return main_out, aux_out, rep_out

        # Mode 2: Standard Training (Supervised / Gen 1+)
        # Returns: (Logits, Aux)
        if self.training:
            return main_out, aux_out

        # Mode 3: Evaluation
        # Returns: Logits
        return main_out

class UniMatch_Model(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super(UniMatch_Model, self).__init__()
        self.encoder = ResNet(Bottleneck, [3, 4, 6, 3], replace_stride_with_dilation=[False, False, True], multi_grid=True, fpn=True)
        self.decoder = dec_deeplabv3_plus(in_planes=self.encoder.get_outplanes(), num_classes=num_classes, low_conv_planes=48, dropout_p=dropout_p)
        self.auxor = Aux_Module(1024, num_classes)

        # Feature Perturbation (Dropout)
        self.fp_drop = nn.Dropout2d(0.5)

    def forward(self, x, feature_perturb=False):
        # Encoder
        feats = self.encoder(x) # [x1, x2, x3, x4]

        # Feature Perturbation (UniPerb)
        if feature_perturb:
            # FIX: Perturb BOTH Low-level (x1) and High-level (x4) features
            # Clone list to avoid side effects
            x1_p = self.fp_drop(feats[0])
            x4_p = self.fp_drop(feats[3])
            feats = [x1_p, feats[1], feats[2], x4_p]

        # Decoder
        main_out = self.decoder(feats)

        # Upsample
        h, w = x.shape[-2:]
        main_out = F.interpolate(main_out, (h, w), mode="bilinear", align_corners=True)

        if self.training and not feature_perturb:
            aux_out = F.interpolate(self.auxor(feats[2]), (h, w), mode="bilinear", align_corners=True)
            return main_out, aux_out

        return main_out


class SimpleDPTHead(nn.Module):
    def __init__(self, in_channels, num_classes, embed_dim=384, dropout_p=0.0):
        super().__init__()

        # 1. Project to common dim
        self.projs = nn.ModuleList([
            nn.Conv2d(in_channels, 256, kernel_size=1) for _ in range(4)
        ])

        # 2. Fuse
        self.fusion = nn.Sequential(
            nn.Conv2d(256 * 4, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.Dropout2d(p=dropout_p),
            nn.Conv2d(512, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
        )

        # 3. New: Upsample Stage (from Stride 14 -> Stride 4)
        # We assume typical input ~798. Stride 14 is ~57. Stride 4 is ~200.
        # We need a learnable upsampling or just simple resize + conv refinement.
        self.upsample_refine = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.Conv2d(128, num_classes, kernel_size=1)
        )

    def forward(self, features):
        # features list: [B, 384, H/14, W/14]
        target_h, target_w = features[0].shape[-2:]

        outs = []
        for i, x in enumerate(features):
            x = self.projs[i](x)
            if x.shape[-2:] != (target_h, target_w):
                x = F.interpolate(x, size=(target_h, target_w), mode='bilinear', align_corners=False)
            outs.append(x)

        x = torch.cat(outs, dim=1)
        x = self.fusion(x) # [B, 256, H/14, W/14]

        # --- FIX: Upsample features to Stride 4 before classifying ---
        # Calculate target size (Input H / 4)
        # We don't have input H here, but we know x is H/14.
        # New size = x_size * (14/4) = x_size * 3.5

        h_high = int(x.shape[-2] * 3.5)
        w_high = int(x.shape[-1] * 3.5)

        x = F.interpolate(x, size=(h_high, w_high), mode='bilinear', align_corners=False)
        logits = self.upsample_refine(x) # [B, num_classes, H/4, W/4]

        return logits

class UniMatchV2_Model(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super(UniMatchV2_Model, self).__init__()
        print("🦕 Loading DINOv2-Small from Torch Hub...")
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.embed_dim = 384

        self.out_indices = [2, 5, 8, 11]

        self.decoder = SimpleDPTHead(self.embed_dim, num_classes, self.embed_dim, dropout_p=dropout_p)

        self.auxor = nn.Sequential(
            nn.Conv2d(self.embed_dim, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, x, return_features=False, complementary_dropout=False):
        B, C, H, W = x.shape

        full_features = self.backbone.get_intermediate_layers(x, n=12, reshape=True)

        # Select the 4 distributed layers
        features_list = [full_features[i] for i in self.out_indices]

        # Complementary Dropout Logic (Same as your original, applied to list)
        if complementary_dropout:
            perturbed_features = []

            # 1. Get dimensions
            # features_list contains [B, C, H, W] tensors
            # We assume input x is [Strong1, Strong2] stacked -> B_total
            last_feat = features_list[-1]
            B_total, C_emb, h_map, w_map = last_feat.shape
            B_half = B_total // 2

            # 2. Generate EXACT 50% mask
            # Paper: "half of its channels (C/2) all set as 1, and others as 0"
            mask = torch.zeros((B_half, C_emb, 1, 1), device=x.device)
            mask[:, :C_emb // 2] = 1.0 # Set first half to 1

            # Shuffle channels to make it random but balanced
            # We shuffle indices along dimension 1
            idx = torch.randperm(C_emb, device=x.device)
            mask = mask[:, idx]

            # 3. Complementary Construction
            # Strong1 gets mask, Strong2 gets (1 - mask)
            full_mask = torch.cat([mask, 1.0 - mask], dim=0)

            # 4. Apply to ALL layers
            for feat in features_list:
                perturbed_features.append(feat * full_mask * 2.0)

            features_list = perturbed_features

        # Decoder
        logits = self.decoder(features_list)

        # Upsample to Image Size (Bilinear)
        logits = F.interpolate(logits, size=(H, W), mode='bilinear', align_corners=False)

        if self.training and not complementary_dropout:
            # Aux head on the LAST layer (index 11)
            aux = self.auxor(features_list[-1])
            aux = F.interpolate(aux, size=(H, W), mode='bilinear', align_corners=False)
            return logits, aux

        return logits


class SegKC_Model(nn.Module):
    def __init__(self, num_classes, dropout_p=0.0):
        super(SegKC_Model, self).__init__()

        # 1. Senior Model (Larger, e.g., ViT-Base)
        print("👴 Building Senior Model (Base)...")
        self.senior = UniMatchV2_Model(num_classes, dropout_p)
        # Force load the Base backbone
        self.senior.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        self.senior.embed_dim = 768
        self.senior.decoder = SimpleDPTHead(768, num_classes, 768, dropout_p)
        self.senior.auxor = nn.Sequential(
            nn.Conv2d(768, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, num_classes, 1)
        )

        # 2. Junior Model (Smaller, e.g., ViT-Small)
        print("👶 Building Junior Model (Small)...")
        # Standard UniMatchV2 init is already ViT-Small
        self.junior = UniMatchV2_Model(num_classes, dropout_p)

    def forward(self, x, return_features=False):
        # We only use the Junior model for inference/validation
        if not self.training:
            return self.junior(x)

        # During training, forward both
        pred_senior = self.senior(x)
        pred_junior = self.junior(x)

        return pred_senior, pred_junior


def build_model(num_classes=HPARAMS["NUM_CLASSES"], dropout_p=0.0):
    method = HPARAMS["SSL_METHOD"]

    if method == "UNIMATCH_V2":
        model = UniMatchV2_Model(num_classes, dropout_p=dropout_p)
    elif method == "U2PL":
        model = U2PL_Model(num_classes, dropout_p=dropout_p)
    elif method == "UNIMATCH":
        model = UniMatch_Model(num_classes, dropout_p=dropout_p)
    elif method == "SEGKC":
        model = SegKC_Model(num_classes, dropout_p=dropout_p)
    else:
        model = iMAS_Model(num_classes, dropout_p=dropout_p)

    # Weight loading for ResNet is skipped for UniMatch V2 (DINOv2 loads its own)
    if method not in ["UNIMATCH_V2", "SEGKC"]:
        w_path = os.path.join(os.path.dirname(DATASET_ROOT), "resnet50d.pth")
        download_backbone_weights(w_path)
        if os.path.exists(w_path):
            print(f"📥 Loading backbone from {w_path}")
            ckpt = torch.load(w_path, map_location="cpu")
            state = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
            dst = model.encoder.state_dict()
            for k, v in state.items():
                if "fc" in k or "classifier" in k: continue
                tk = k.replace("downsample.1", "downsample.0").replace("downsample.2", "downsample.1")
                if tk in dst and dst[tk].shape == v.shape: dst[tk].copy_(v)

    return model


# --- D. Datasets & Transforms ---

class iMASAug:
    """
    iMAS-specific dual augmentation pipeline (Discrete Ops + CutMix support).
    Renamed from WeakStrongAug. Identical logic + Fix for padding.
    """
    def __init__(self):
        self.norm = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True),
                                v2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        self.pool = [
            lambda i: i, # Identity
            lambda i: ImageOps.invert(i.convert('RGB')),
            lambda i: ImageOps.autocontrast(i),
            lambda i: ImageOps.equalize(i),
            lambda i: i.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.1, 2.0))),
            lambda i: ImageEnhance.Contrast(i).enhance(random.uniform(0.1, 0.95)),
            lambda i: ImageEnhance.Sharpness(i).enhance(random.uniform(0.1, 0.95)),
            lambda i: ImageEnhance.Color(i).enhance(random.uniform(0.1, 0.95)),
            lambda i: ImageEnhance.Brightness(i).enhance(random.uniform(0.1, 0.95)),
            lambda i: ImageOps.posterize(i.convert('RGB'), random.randint(4, 8)),
            lambda i: ImageOps.solarize(i.convert('RGB'), random.randint(1, 255)),
            lambda i: v2.functional.to_pil_image(v2.functional.adjust_hue(v2.functional.to_image(i), random.uniform(-0.1, 0.1)))
        ]

    def __call__(self, img, mask=None):
        # 1. Spatial (Weak)
        s = random.uniform(HPARAMS["SCALE_MIN"], HPARAMS["SCALE_MAX"])
        h, w = int(img.height * s), int(img.width * s)
        img = Fv2.resize(img, (h, w), InterpolationMode.BILINEAR)
        if mask: mask = Fv2.resize(mask, (h, w), InterpolationMode.NEAREST)

        cs = HPARAMS["CROP_SIZE"]
        if h < cs or w < cs:
            pad_h, pad_w = max(0, cs - h), max(0, cs - w)
            fill_color = (123, 116, 103)
            img = Fv2.pad(img, [0, 0, pad_w, pad_h], fill=fill_color)
            if mask: mask = Fv2.pad(mask, [0, 0, pad_w, pad_h], fill=HPARAMS["IGNORE_INDEX"])

        i, j, h, w = v2.RandomCrop.get_params(img, (cs, cs))
        img = Fv2.crop(img, i, j, h, w)
        if mask: mask = Fv2.crop(mask, i, j, h, w)

        if random.random() < 0.5:
            img = Fv2.hflip(img)
            if mask: mask = Fv2.hflip(mask)

        # 2. Strong
        img_s = img
        for op in random.sample(self.pool, 2): img_s = op(img_s)

        res = {'weak': self.norm(img), 'strong': self.norm(img_s)}
        if mask: res['mask'] = torch.from_numpy(np.array(mask, dtype=np.uint8)).long()
        return res

class StandardColorAug:
    """
    Standard SSL Augmentation (UniMatch / U2PL / FixMatch).
    Strong Aug = ColorJitter + Grayscale + GaussianBlur.
    mode='dual' returns {'weak', 'strong1', 'strong2'} (for UniMatch).
    mode='single' returns {'weak', 'strong'} (for U2PL).
    """
    def __init__(self, mode="single", strength="standard"):
        self.mode = mode
        self.norm = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True),
                                v2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

        # Configure Jitter based on Method
        if strength == "strong": # UniMatch / ST++
            jitter_params = (0.8, 0.8, 0.8, 0.2)
        else: # U2PL / Standard SimCLR
            jitter_params = (0.4, 0.4, 0.4, 0.1)

        self.strong_transforms = v2.Compose([
            v2.RandomApply([v2.ColorJitter(*jitter_params)], p=0.8),
            v2.RandomGrayscale(p=0.2),
            v2.RandomApply([v2.GaussianBlur(kernel_size=(5,5))], p=0.5)
        ])

    def get_strong(self, img):
        return self.strong_transforms(img)

    def __call__(self, img, mask=None):
        # 1. Weak Spatial (Same as iMAS logic for fair comparison)
        s = random.uniform(HPARAMS["SCALE_MIN"], HPARAMS["SCALE_MAX"])
        h, w = int(img.height * s), int(img.width * s)
        img = Fv2.resize(img, (h, w), InterpolationMode.BILINEAR)
        if mask: mask = Fv2.resize(mask, (h, w), InterpolationMode.NEAREST)

        cs = HPARAMS["CROP_SIZE"]
        if h < cs or w < cs:
            pad_h, pad_w = max(0, cs - h), max(0, cs - w)
            fill_color = (123, 116, 103)
            img = Fv2.pad(img, [0, 0, pad_w, pad_h], fill=fill_color)
            if mask: mask = Fv2.pad(mask, [0, 0, pad_w, pad_h], fill=HPARAMS["IGNORE_INDEX"])

        i, j, h, w = v2.RandomCrop.get_params(img, (cs, cs))
        img = Fv2.crop(img, i, j, h, w)
        if mask: mask = Fv2.crop(mask, i, j, h, w)

        if random.random() < 0.5:
            img = Fv2.hflip(img)
            if mask: mask = Fv2.hflip(mask)

        # 2. Strong Intensity
        img_weak = self.norm(img)

        if self.mode == "dual":
            s1 = self.get_strong(img)
            s2 = self.get_strong(img)
            res = {'weak': img_weak, 'strong1': self.norm(s1), 'strong2': self.norm(s2)}
        else:
            s1 = self.get_strong(img)
            res = {'weak': img_weak, 'strong': self.norm(s1)}

        if mask: res['mask'] = torch.from_numpy(np.array(mask, dtype=np.uint8)).long()
        return res

class BaseTrainDataset(Dataset):
    def __init__(self, image_paths_file, id_to_trainid_map=None):
        with open(image_paths_file, 'r') as f: self.image_paths = f.read().splitlines()
        self.id_to_trainid = id_to_trainid_map

    def __len__(self): return len(self.image_paths)

    def apply_spatial_common(self, img, mask, smin, smax):
        """Shared logic for Resize -> Pad -> Crop -> Flip"""
        s = random.uniform(smin, smax)
        h, w = int(img.height * s), int(img.width * s)
        img = Fv2.resize(img, (h, w), InterpolationMode.BILINEAR)
        if mask: mask = Fv2.resize(mask, (h, w), InterpolationMode.NEAREST)

        cs = HPARAMS["CROP_SIZE"]
        pad_h, pad_w = max(0, cs - h), max(0, cs - w)
        if pad_h > 0 or pad_w > 0:
            fill_color = (123, 116, 103)
            img = Fv2.pad(img, [0, 0, pad_w, pad_h], fill=fill_color)
            if mask: mask = Fv2.pad(mask, [0, 0, pad_w, pad_h], fill=HPARAMS["IGNORE_INDEX"])

        i, j, h, w = v2.RandomCrop.get_params(img, (cs, cs))
        img = Fv2.crop(img, i, j, h, w)
        if mask: mask = Fv2.crop(mask, i, j, h, w)

        if random.random() < 0.5:
            img = Fv2.hflip(img)
            if mask: mask = Fv2.hflip(mask)

        return img, mask

class CityscapesLabeled(BaseTrainDataset):
    """
    Handles Labeled Data.
    Mode 'ssl' delegates to a pluggable ssl_aug_strategy (iMAS, UniMatch, etc).
    Mode 'std' returns (Image, Label) for Supervised/GenX.
    """
    def __init__(self, image_paths_file, id_to_trainid_map, mode="ssl", aug_provider=None, ssl_aug_strategy=None):
        super().__init__(image_paths_file, id_to_trainid_map)
        self.mode = mode
        self.ext_aug = aug_provider

        # Default to iMASAug if not provided (Backward compatibility)
        if ssl_aug_strategy is None:
            self.ssl_aug = iMASAug()
        else:
            self.ssl_aug = ssl_aug_strategy

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        d, n = os.path.split(path)
        m_path = os.path.join(d.replace('leftImg8bit', 'gtFine'), n.replace('_leftImg8bit.png', '_gtFine_labelIds.png'))

        img = Image.open(path).convert("RGB")
        mask = Image.open(m_path)

        if self.mode == "ssl":
            # Delegate to the strategy class
            res = self.ssl_aug(img, mask)

            # Map mask
            mask_np = res['mask'].numpy()
            mask_mapped = self.id_to_trainid[mask_np]

            res['mask'] = torch.from_numpy(mask_mapped).long()
            res['flag'] = torch.tensor(0)
            return res
        else:
            # Return Standard single image for Supervised/GenX
            if self.ext_aug:
                (smin, smax), tr = self.ext_aug["get_scale"](), self.ext_aug["get_transforms"]()
            else:
                smin, smax = 0.5, 2.0
                tr = v2.Compose([
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, True),
                    v2.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
                ])

            img, mask = self.apply_spatial_common(img, mask, smin, smax)
            return {
                'image': tr(img),
                'mask': torch.from_numpy(self.id_to_trainid[np.array(mask, dtype=np.uint8)]).long(),
                'flag': torch.tensor(0)
            }

class CityscapesUnlabeled(BaseTrainDataset):
    """Unlabeled Data for SSL. Delegates to ssl_aug_strategy."""
    def __init__(self, image_paths_file, ssl_aug_strategy=None):
        super().__init__(image_paths_file, None)
        self.ssl_aug = ssl_aug_strategy if ssl_aug_strategy else iMASAug()

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        res = self.ssl_aug(img, None)
        res['flag'] = torch.tensor(1) # Flag 1 for unlabeled
        return res

class CityscapesPseudo(BaseTrainDataset):
    """
    For Gen 1+ Self-Training.
    Loads masks from a pseudo-label directory (already TrainIDs).
    Zeroes out image pixels where mask == IGNORE_INDEX (Inconsistency).
    """
    def __init__(self, image_paths_file, pseudo_dir, aug_provider):
        super().__init__(image_paths_file, None)
        self.pseudo_dir = pseudo_dir
        self.aug_provider = aug_provider

    def _get_aug_params(self):
        return self.aug_provider["get_scale"](), self.aug_provider["get_transforms"]()

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        mask_name = os.path.basename(path).replace('_leftImg8bit.png','_gtFine_labelIds.png')
        m_path = os.path.join(self.pseudo_dir, mask_name)

        img = Image.open(path).convert("RGB")
        mask = Image.open(m_path) # Already TrainIDs

        (smin, smax), tr = self._get_aug_params()
        img, mask = self.apply_spatial_common(img, mask, smin, smax)

        img_t = tr(img)
        mask_t = torch.from_numpy(np.array(mask, dtype=np.uint8)).long()

        # Zero out inconsistent regions (IGNORE_INDEX, usually 255)
        # We use HPARAMS["IGNORE_INDEX"] implicitly as we expect mask to have 255 where invalid
        # To be safe, we check against the config constant
        ignore_val = HPARAMS["IGNORE_INDEX"]
        is_inconsistent = (mask_t == ignore_val)

        # Expand mask to (3, H, W) to match image channels
        img_t[:, is_inconsistent] = 0.0

        return {
            'image': img_t,
            'mask': mask_t,
            'flag': torch.tensor(1)
        }

class CityscapesPseudoStrong(CityscapesPseudo):
    """
    Variant of PseudoDataset that specifically requests STRONG augmentations.
    """
    def _get_aug_params(self):
        if "get_transforms_strong" in self.aug_provider:
            return self.aug_provider["get_scale"](), self.aug_provider["get_transforms_strong"]()
        return super()._get_aug_params()

class ValDataset(Dataset):
    """Simple Validation Dataset"""
    def __init__(self, image_paths_file, id_to_trainid_map):
        with open(image_paths_file, 'r') as f: self.image_paths = f.read().splitlines()
        self.id_to_trainid = id_to_trainid_map
        self.norm = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True),
                                v2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        d, n = os.path.split(path)
        m_path = os.path.join(d.replace('leftImg8bit','gtFine'), n.replace('_leftImg8bit.png','_gtFine_labelIds.png'))

        img = Image.open(path).convert("RGB")
        mask = Image.open(m_path)

        return self.norm(img), torch.from_numpy(self.id_to_trainid[np.array(mask, dtype=np.uint8)]).long()

In [ ]:
# ==============================================================================
# --- 4a) CORE INFRASTRUCTURE: UTILS & BASE TRAINER ---
# ==============================================================================

# --- Utilities (EMA, Optim, Scheduler, Weights) ---
@torch.no_grad()
def ema_update(teacher, student, decay):
    for t_param, s_param in zip(teacher.parameters(), student.parameters()):
        t_param.data.mul_(decay).add_(s_param.data, alpha=1.0 - decay)
    for t_buf, s_buf in zip(teacher.buffers(), student.buffers()):
        if torch.is_floating_point(t_buf):
            t_buf.data.mul_(decay).add_(s_buf.data, alpha=1.0 - decay)
        else:
            t_buf.data.copy_(s_buf.data)

def get_optimizer(model, lr_factor=1.0):
    opt_type = HPARAMS["ACTIVE_OPTIMIZER"]
    cfg      = HPARAMS["OPTIMIZERS"][opt_type]

    lr_head  = cfg["LR_HEAD"] * lr_factor
    lr_bb    = cfg["LR_BACKBONE"] * lr_factor
    weight_decay = cfg.get("WEIGHT_DECAY", 0.01)

    # Helper to identify backbone parameters vs head parameters
    # ResNet uses 'encoder', DINOv2 uses 'backbone'
    def is_backbone(name):
        return 'encoder' in name or 'backbone' in name

    params = [
        # Backbone Weights (Decay)
        {'params': [p for n, p in model.named_parameters() if is_backbone(n) and p.dim() > 1],
         'lr': lr_bb, 'weight_decay': weight_decay},

        # Backbone Biases/Norms (No Decay)
        {'params': [p for n, p in model.named_parameters() if is_backbone(n) and p.dim() <= 1],
         'lr': lr_bb, 'weight_decay': 0.0},

        # Head Weights (Decay)
        {'params': [p for n, p in model.named_parameters() if not is_backbone(n) and p.dim() > 1],
         'lr': lr_head, 'weight_decay': weight_decay},

        # Head Biases/Norms (No Decay)
        {'params': [p for n, p in model.named_parameters() if not is_backbone(n) and p.dim() <= 1],
         'lr': lr_head, 'weight_decay': 0.0},
    ]

    if "SGD" in opt_type:
        return torch.optim.SGD(params, momentum=cfg["MOMENTUM"], nesterov=cfg["NESTEROV"])
    elif "ADAMW" in opt_type:
        return torch.optim.AdamW(params, betas=cfg.get("BETAS", (0.9, 0.999)))
    elif "ADAM" in opt_type:
        return torch.optim.Adam(params, betas=cfg.get("BETAS", (0.9, 0.999)))

class WarmupPolyLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, max_iters, warmup_iters=1500, power=0.9, min_scale=0.01, last_epoch=-1):
        self.max_iters = max_iters; self.warmup_iters = warmup_iters; self.power = power; self.min_scale = min_scale
        super().__init__(optimizer, last_epoch)
    def get_lr(self):
        step = self.last_epoch + 1
        if self.warmup_iters > 0 and step <= self.warmup_iters: scale = step / self.warmup_iters
        else:
            denom = max(1, self.max_iters - self.warmup_iters)
            scale = max(0.0, 1.0 - ((step - self.warmup_iters) / denom)) ** self.power
        return [base_lr * max(scale, self.min_scale) for base_lr in self.base_lrs]

def compute_class_weights(file_path, id_map):
    print(f"⚖️ Computing weights from {file_path}...")
    num_classes = HPARAMS["NUM_CLASSES"]
    counts = np.zeros(num_classes, dtype=np.int64)

    with open(file_path, 'r') as f: lines = f.read().splitlines()
    found = 0

    for p in tqdm(lines, desc="Weight Scan", leave=False):
        img_dir, img_name = os.path.split(p)
        mask_dir = img_dir.replace('leftImg8bit', 'gtFine')
        mask_name = img_name.replace('_leftImg8bit.png', '_gtFine_labelIds.png')
        m_path = os.path.join(mask_dir, mask_name)

        try:
            if not os.path.exists(m_path): continue
            m = np.array(Image.open(m_path), dtype=np.uint8)
            train_ids = id_map[m]
            u, c = np.unique(train_ids, return_counts=True)
            for k, v in zip(u, c):
                # Only count valid classes (0-18)
                if k < num_classes:
                    counts[k] += v
            found += 1
        except: pass

    if found == 0: return torch.ones(num_classes, dtype=torch.float32)

    # Calculate Weights (Inverse Frequency)
    # We enforce a min count of 1 to avoid division by zero
    counts = np.maximum(counts, 1)
    freqs = counts.astype(np.float64)
    med = np.median(freqs)
    weights = med / freqs

    # Normalize so mean is 1.0
    weights /= max(1e-8, weights.mean())

    print(f"✅ Computed weights over {found} images.")
    return torch.tensor(weights, dtype=torch.float32)


def load_checkpoint_for_inference(model, checkpoint_path, device):
    """
    Robustly loads weights. Handles SegKC checkpoints by extracting Junior weights.
    """
    ckpt = torch.load(checkpoint_path, map_location=device)
    state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt

    new_state_dict = {}

    # Check if this is a SegKC checkpoint (contains 'junior.' or 'senior.')
    keys = list(state_dict.keys())
    is_segkc = any('junior.' in k for k in keys)

    if is_segkc:
        print(f"   🔄 SegKC Checkpoint detected. Extracting 'junior' (Small) weights...")
        for k, v in state_dict.items():
            if k.startswith('junior.'):
                # Remove 'junior.' prefix: 'junior.backbone.x' -> 'backbone.x'
                new_key = k[7:]
                new_state_dict[new_key] = v
            # Ignore senior weights
    else:
        # Check if we are loading a Senior (Base) checkpoint into a Small model
        # Base cls_token is [1, 1, 768], Small is [1, 1, 384]
        if 'backbone.cls_token' in state_dict and state_dict['backbone.cls_token'].shape[-1] == 768:
             print("   ❌ ERROR: Attempting to load ViT-Base weights into ViT-Small model!")
             print("   This likely means you are trying to use the SegKC 'Teacher' checkpoint.")
             print("   Please use the 'Student' checkpoint for Gen 1+ generation.")
             raise RuntimeError("Dimension mismatch: Checkpoint is Base (768), Model is Small (384).")

        new_state_dict = state_dict


    # Add this check before loading
    if 'backbone.cls_token' in new_state_dict:
        ckpt_dim = new_state_dict['backbone.cls_token'].shape[-1]
        model_dim = model.embed_dim # Ensure your model class has this attribute

        if ckpt_dim != model_dim:
            print(f"❌ SIZE MISMATCH: Checkpoint ({ckpt_dim}) vs Model ({model_dim})")
            print("   You are likely trying to load a Senior (Base) checkpoint into a Junior (Small) model.")
            print("   We cannot proceed with this checkpoint. Re-initializing from scratch (or skip loading).")
            return # Skip loading to prevent crash

    # Load
    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
    # Filter out harmless missing keys
    real_missing = [k for k in missing if 'num_batches_tracked' not in k]
    if len(real_missing) > 0:
        print(f"   ⚠️ Missing keys: {len(real_missing)} (e.g. {real_missing[:2]})")


def _lovasz_grad(gt_sorted):
    """
    Computes gradient of the Jaccard extension w.r.t the sort
    """
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1. - intersection / union
    if p > 1: # cover 1-pixel case
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard

def _lovasz_softmax_flat(probs, labels, ignore_index):
    """
    Multi-class Lovasz-Softmax loss
    """
    if probs.numel() == 0:
        return torch.tensor(0.0).to(probs.device)

    C = probs.size(1)
    losses = []

    # Filter valid pixels
    valid = (labels != ignore_index)
    if valid.sum() == 0:
        return probs.sum() * 0.

    probs = probs[valid]
    labels = labels[valid]

    for c in range(C):
        fg = (labels == c).float() # foreground for class c
        if fg.sum() == 0:
            continue

        class_pred = probs[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, 0, descending=True)
        perm = perm.data
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))

    if len(losses) > 0:
        return torch.stack(losses).mean()
    else:
        return probs.sum() * 0.

def lovasz_softmax_loss_custom(logits, labels, ignore_index=255):
    """
    logits: [B, C, H, W]
    labels: [B, H, W]
    """
    probs = F.softmax(logits, dim=1)
    # Flatten: [B, C, H, W] -> [P, C]
    probs = probs.permute(0, 2, 3, 1).contiguous().view(-1, logits.size(1))
    labels = labels.view(-1)
    return _lovasz_softmax_flat(probs, labels, ignore_index)

def make_loss_fn(loss_type: str, class_weights: torch.Tensor | None):
    """Factory for loss functions. Ignores OHEM params if not using OHEM loss."""
    loss_type = loss_type.lower()
    ignore = HPARAMS["IGNORE_INDEX"]

    def ohem_loss(logits, targets, valid_mask=None):
        # 1. Standard NLL
        nll = F.cross_entropy(logits, targets, ignore_index=ignore, reduction='none')

        # 2. Get Probabilities for Thresholding
        with torch.no_grad():
            probs = F.softmax(logits, dim=1)
            # Gather probability of the correct class
            # We must mask the target for gather to work (replace ignore 19 with 0)
            safe_targets = targets.clone()
            safe_targets[targets == ignore] = 0
            p_correct = probs.gather(1, safe_targets.unsqueeze(1)).squeeze(1)

        # 3. Create Mask: Hard pixels (prob < thresh) OR Valid pixels (not ignore)
        # The log says thresh 0.7. So pixels with p_correct < 0.7 are "Hard".
        thresh = HPARAMS.get("OHEM_THRESH", 0.7)
        min_kept = int(HPARAMS.get("OHEM_MIN_KEPT", 100000))

        # Valid pixels (not ignore)
        valid_pix = (targets != ignore)

        # Hard pixels
        hard_pix = (p_correct < thresh) & valid_pix

        # 4. Enforce Min Kept
        # If we have fewer hard pixels than min_kept, we take the top-k highest loss pixels
        num_hard = hard_pix.sum()

        if num_hard < min_kept:
            # Flatten losses on valid pixels
            valid_nll = nll[valid_pix]
            # If total valid pixels < min_kept, take all valid pixels
            k = min(int(valid_nll.numel()), min_kept)

            if k > 0:
                # Top-k highest losses
                _, topk_indices = torch.topk(valid_nll, k)
                return valid_nll[topk_indices].mean()
            else:
                return nll.sum() * 0.0 # No valid pixels
        else:
            # We have enough hard pixels, just use them
            return nll[hard_pix].mean()

    if loss_type == "ce":
        return nn.CrossEntropyLoss(ignore_index=ignore)

    if loss_type == "dice":
        return lambda p, t: smp.losses.DiceLoss(mode='multiclass', ignore_index=ignore)(p, t)

    if loss_type == "ce_lovasz":
        ce = nn.CrossEntropyLoss(ignore_index=ignore)

        def criterion(logits, targets, valid_mask=None):
            loss_ce = ce(logits, targets)

            loss_lov = lovasz_softmax_loss_custom(logits, targets, ignore)
            return loss_ce + 0.5 * loss_lov

        return criterion

    if loss_type == "wce_lovasz":
        wce = nn.CrossEntropyLoss(ignore_index=ignore, weight=class_weights, reduction='mean')

        def criterion(logits, targets, valid_mask=None):
            loss_ce = wce(logits, targets)

            loss_lov = lovasz_softmax_loss_custom(logits, targets, ignore)
            return loss_ce + 0.5 * loss_lov

        return criterion

    if loss_type == "ohem":
        return ohem_loss

    # Fallback for complex legacy losses (OHEM, etc - retained as requested)
    print(f"⚠️ Warning: Loss '{loss_type}' not fully optimized in cleaned script. Using CE.")
    return nn.CrossEntropyLoss(ignore_index=ignore)

# --- Validation Helpers ---
@torch.inference_mode()
def validate_miou_and_ce(val_loader, model, device):
    model.eval()
    N_VALID = HPARAMS["NUM_CLASSES"] # 19
    IGN = HPARAMS["IGNORE_INDEX"]    # 255

    hist = np.zeros((N_VALID, N_VALID), dtype=np.int64)
    loss_sum, pix_cnt = 0.0, 0

    for images, targets in val_loader:
        images, targets = images.to(device), targets.to(device)
        with amp.autocast(device_type=device):
            logits = model(images)
            if isinstance(logits, tuple): logits = logits[0]
            loss_sum += F.cross_entropy(logits, targets, ignore_index=IGN, reduction='sum').item()

        # --- FIX: Strict filtering ---
        # Ensure we only count pixels that are NOT ignore index AND within valid range (0-18)
        valid = (targets != IGN) & (targets < N_VALID)

        pix_cnt += valid.sum().item()
        if valid.any():
            pred = logits.argmax(1)
            # Safe numpy conversion
            t_np = targets[valid].cpu().numpy()
            p_np = pred[valid].cpu().numpy()

            # Bincount
            hist += np.bincount(t_np * N_VALID + p_np, minlength=N_VALID**2).reshape(N_VALID, N_VALID)

    miou = np.nanmean(np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist) + 1e-10))
    return miou, loss_sum/max(1, pix_cnt), np.diag(hist)/(hist.sum(1)+hist.sum(0)-np.diag(hist)+1e-10)

@torch.inference_mode()
def validate_sliding(loader, model, device, tile_size=None, overlap=None):
    """Full Sliding Window Inference."""
    if tile_size is None:
        tile_size = HPARAMS["CROP_SIZE"]
    if overlap is None:
        overlap = HPARAMS["SW_OVERLAP"]

    model.eval()
    N_CLS = HPARAMS["NUM_CLASSES"]
    hist = np.zeros((N_CLS, N_CLS))
    stride = int(tile_size * (1 - overlap))
    for img, label in tqdm(loader, desc="Val (Sliding)", leave=False):
        img = img.to(device)
        B, C, H, W = img.shape
        full_probs = torch.zeros((B, N_CLS, H, W), device=device)
        count_map = torch.zeros((B, 1, H, W), device=device)
        rows = int(math.ceil((H - tile_size) / stride) + 1)
        cols = int(math.ceil((W - tile_size) / stride) + 1)
        for r in range(rows):
            for c in range(cols):
                y1 = min(r * stride, H - tile_size)
                x1 = min(c * stride, W - tile_size)
                y2, x2 = y1 + tile_size, x1 + tile_size
                tile = img[:, :, y1:y2, x1:x2]
                with amp.autocast(device_type=device):
                    out = model(tile)
                    if isinstance(out, tuple): out = out[0]
                full_probs[:, :, y1:y2, x1:x2] += out
                count_map[:, :, y1:y2, x1:x2] += 1.0
        full_probs /= count_map
        preds = full_probs.argmax(1).cpu().numpy()
        label = label.numpy()
        valid = (label != HPARAMS["IGNORE_INDEX"])
        if valid.any():
            hist += np.bincount(label[valid] * N_CLS + preds[valid], minlength=N_CLS**2).reshape(N_CLS, N_CLS)
    iou = np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist) + 1e-10)
    return float(np.nanmean(iou[:19])), iou[:19]

# --- Base Class ---
class BaseSSLTrainer:
    """Abstract Base Class. Handles the loop, logging, and EMA."""
    def __init__(self, model, teacher, optimizer, scheduler, scaler, loss_fn, device, algorithm_config):
        self.model = model
        self.teacher = teacher
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.scaler = scaler
        self.loss_fn = loss_fn
        self.device = device
        self.cfg = algorithm_config
        self.current_step = 0
        self.metric_name = getattr(self, "_METRIC_NAME", "Metric")

    def train_epoch(self, loader_l, loader_u=None):
        self.model.train()
        if self.teacher: self.teacher.eval()

        if loader_u is not None:
            iterator = tqdm(loader_u, desc=f"Train ({HPARAMS['SSL_METHOD']})", leave=False)
            iter_l = iter(loader_l)
            is_ssl = True
        else:
            iterator = tqdm(loader_l, desc="Train (Sup)", leave=False)
            is_ssl = False

        avg_loss = 0.0; avg_metric = 0.0

        for batch_primary in iterator:
            batch_u = batch_primary if is_ssl else None
            batch_l = None
            if is_ssl:
                try: batch_l = next(iter_l)
                except StopIteration:
                    iter_l = iter(loader_l)
                    batch_l = next(iter_l)
            else:
                batch_l = batch_primary

            if batch_l: batch_l = {k: v.to(self.device, non_blocking=True) for k, v in batch_l.items()}
            if batch_u: batch_u = {k: v.to(self.device, non_blocking=True) for k, v in batch_u.items()}

            self.optimizer.zero_grad(set_to_none=True)
            loss, metric = self.train_step(batch_l, batch_u)

            if torch.isfinite(loss):
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()

                if self.teacher:
                    global_step = (self.scheduler.last_epoch if self.scheduler else self.current_step) + 1
                    target_alpha = self.cfg.get("EMA_ALPHA", 0.999)
                    curr_alpha = min(1.0 - 1.0 / (global_step + 1), target_alpha)
                    ema_update(self.teacher, self.model, curr_alpha)

                if self.scheduler: self.scheduler.step()

                avg_loss += loss.item()
                if metric: avg_metric += metric.item()
                self.current_step += 1

        return avg_loss / max(1, len(iterator)), avg_metric / max(1, len(iterator))

    def train_step(self, batch_l, batch_u):
        raise NotImplementedError

In [ ]:
# ==============================================================================
# --- 4b) STRATEGY: SUPERVISED BASELINE ---
# ==============================================================================

class SupervisedTrainer(BaseSSLTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.prob_cutmix = self.cfg.get("CUTMIX_PROB", 0.0)
        self.prob_classmix = self.cfg.get("CLASSMIX_PROB", 0.0)

        # DEFINITION: Thing vs Stuff
        # Stuff (Background): Road(0), SW(1), Build(2), Wall(3), Fence(4), Veg(8), Terrain(9), Sky(10)
        # Things (Foreground): Pole(5), TL(6), TS(7), Person(11), Rider(12), Car(13), Truck(14), Bus(15), Train(16), Motor(17), Bike(18)

        # We allow mixing of ALL Things + Wall/Fence (sometimes useful)
        # We strictly EXCLUDE: Road, Sidewalk, Building, Vegetation, Terrain, Sky
        self.mixable_classes = torch.tensor([3, 4, 5, 6, 7, 11, 12, 13, 14, 15, 16, 17, 18], device=self.device)

    def apply_cutmix(self, images, targets):
        """Standard Box-based CutMix"""
        B, C, H, W = images.shape
        perm = torch.randperm(B, device=self.device)
        imgs_partner = images[perm]
        lbls_partner = targets[perm]

        lam = np.random.beta(1.0, 1.0)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        images[:, :, bby1:bby2, bbx1:bbx2] = imgs_partner[:, :, bby1:bby2, bbx1:bbx2]
        targets[:, bby1:bby2, bbx1:bbx2] = lbls_partner[:, bby1:bby2, bbx1:bbx2]
        return images, targets

    def apply_classmix(self, images, targets):
        """
        Foreground-Only ClassMix.
        Only pastes objects (Things) onto the target image to preserve scene geometry.
        """
        B, C, H, W = images.shape
        perm = torch.randperm(B, device=self.device)
        imgs_partner = images[perm]
        lbls_partner = targets[perm]

        mix_mask = torch.zeros_like(targets, dtype=torch.bool)

        for i in range(B):
            # 1. Identify classes in the partner image
            classes = torch.unique(lbls_partner[i])
            classes = classes[classes != HPARAMS["IGNORE_INDEX"]]

            if len(classes) == 0: continue

            # 2. Filter: Only allow 'Mixable' classes (Foreground)
            is_mixable = torch.isin(classes, self.mixable_classes)
            valid_classes = classes[is_mixable]

            if len(valid_classes) == 0: continue

            # 3. Selection: Randomly pick 50% of the valid foreground objects
            n_select = (len(valid_classes) + 1) // 2
            selected = valid_classes[torch.randperm(len(valid_classes))[:n_select]]

            # 4. Create Mask
            for c in selected:
                mix_mask[i] |= (lbls_partner[i] == c)

        # Apply Mix
        mix_mask_img = mix_mask.unsqueeze(1).expand_as(images)
        images[mix_mask_img] = imgs_partner[mix_mask_img]
        targets[mix_mask] = lbls_partner[mix_mask]

        return images, targets

    def train_step(self, batch_l, batch_u):
        img = batch_l.get('image', batch_l.get('weak'))
        y = batch_l['mask']

        r = random.random()

        if r < self.prob_classmix:
            img, y = self.apply_classmix(img, y)
        elif r < (self.prob_classmix + self.prob_cutmix):
            img, y = self.apply_cutmix(img, y)

        with amp.autocast(device_type=self.device):
            logits, aux = self.model(img)
            loss = self.loss_fn(logits, y)
            if aux is not None:
                aux = F.interpolate(aux, size=y.shape[-2:], mode='bilinear', align_corners=False)
                loss += 0.4 * F.cross_entropy(aux, y, ignore_index=HPARAMS["IGNORE_INDEX"])

        return loss, torch.tensor(0.0)

In [ ]:
# ==============================================================================
# --- 4c) STRATEGY: iMAS (Instance-Aware Mixed Augmentation) ---
# ==============================================================================


def compute_instance_mean_iou(pred, target, num_classes=20, ignore_index=19):
    B = pred.shape[0]
    ious = []
    pred = pred.view(B, -1)
    target = target.view(B, -1)
    for i in range(B):
        p_i, t_i = pred[i], target[i]
        valid = (t_i != ignore_index)
        p_i, t_i = p_i[valid], t_i[valid]
        if len(t_i) == 0:
            ious.append(0.0); continue

        hist = torch.bincount(t_i * num_classes + p_i, minlength=num_classes**2).float()
        hist = hist.view(num_classes, num_classes)
        intersection = torch.diag(hist)
        union = hist.sum(1) + hist.sum(0) - intersection
        iou_cls = intersection / (union + 1e-6)

        weights = hist.sum(1)
        weights[0] = 0.0 # Ignore background/road? (From official code)
        if weights.sum() > 0:
            miou = (iou_cls * (weights / weights.sum())).sum()
        else: miou = torch.tensor(1.0, device=pred.device)
        ious.append(miou)
    return torch.stack(ious)

@torch.no_grad()
def compute_ulb_hardness_official(stu_logits, tea_logits, thresh=0.95):
    """Exact port of iMAS hardness calculation."""
    num_classes = stu_logits.shape[1]
    prob_stu = torch.softmax(stu_logits, dim=1)
    prob_tea = torch.softmax(tea_logits, dim=1)

    max_prob_stu, pred_labl_stu = torch.max(prob_stu, dim=1)
    max_prob_tea, pred_labl_tea = torch.max(prob_tea, dim=1)

    # 1. IoU consistency
    hardness_iou_tea = compute_instance_mean_iou(pred_labl_stu, pred_labl_tea, num_classes=num_classes)
    hardness_iou_stu = compute_instance_mean_iou(pred_labl_tea, pred_labl_stu, num_classes=num_classes)

    # 2. Confidence Ratio
    hardness_ratio_tea = max_prob_tea.ge(thresh).float().mean(dim=[1,2])
    hardness_ratio_stu = max_prob_stu.ge(thresh).float().mean(dim=[1,2])

    # 3. Combine
    tmp_ratio = hardness_ratio_tea + hardness_ratio_stu + 1e-6
    hardness_v2 = (hardness_ratio_tea / tmp_ratio) * hardness_iou_tea + \
                  (hardness_ratio_stu / tmp_ratio) * hardness_iou_stu

    return hardness_v2

# --- CutMix Helpers ---
def get_adaptive_magnitude(v_min, v_max, confidence, flag_tough_max=True):
    if flag_tough_max: var_mu = v_min + (v_max - v_min) * confidence
    else: var_mu = v_max - (v_max - v_min) * confidence
    var_sigma = max(var_mu - v_min, v_max - var_mu) / 3.0
    a, b = (v_min - var_mu) / (var_sigma + 1e-6), (v_max - var_mu) / (var_sigma + 1e-6)
    return stats.truncnorm(a, b, loc=var_mu, scale=var_sigma + 1e-6).rvs()

def rand_bbox_adaptive(size, v_ratio):
    B, C, H, W = size
    if isinstance(v_ratio, list): v_ratio = np.array(v_ratio)
    cut_w = (W * v_ratio).astype(int); cut_h = (H * v_ratio).astype(int)
    cx = np.random.randint(W, size=B); cy = np.random.randint(H, size=B)
    bbx1 = np.clip(cx - cut_w // 2, 0, W); bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W); bby2 = np.clip(cy + cut_h // 2, 0, H)
    return bbx1, bby1, bbx2, bby2


class IMASTrainer(BaseSSLTrainer):
    _METRIC_NAME = "Hardness"

    def train_step(self, batch_l, batch_u):
        xw_l, y_l = batch_l['weak'], batch_l['mask']
        xw_u, xs_u = batch_u['weak'], batch_u['strong']

        conf_tau = self.cfg["CONF_TAU"]
        lambda_u = self.cfg["LAMBDA_U"]

        with torch.no_grad():
            logits_t = self.teacher(xw_u)
            logits_s_weak, _ = self.model(xw_u)

            probs_t = F.softmax(logits_t, dim=1)
            conf_t, pl_t = probs_t.max(1)

            # iMAS Hardness: High (1.0) = Hard, Low (0.0) = Easy
            hardness = compute_ulb_hardness_official(logits_s_weak, logits_t, thresh=conf_tau)

            # --- HYBRID FIX: Use Hardness for Mixing (Eq 8) ---
            gamma_mix = hardness.view(-1, 1, 1, 1)

        # A. Intensity Mix
        xs_intensity = (gamma_mix * xs_u) + ((1.0 - gamma_mix) * xw_u)

        # B. CutMix Setup
        xs_cutmix = xs_intensity.clone()
        pl_cutmix = pl_t.clone()
        conf_cutmix = conf_t.clone()

        # Initialize Spatial Weight Map using HARDNESS (Your preferred logic)
        B, C, H, W = xs_u.shape
        weight_map = hardness.view(B, 1, 1, 1).expand(B, 1, H, W).clone()

        if self.cfg.get("USE_CUTMIX", True):
            # Sort: Easy -> Hard
            sorted_indices = torch.argsort(hardness)

            idx_source = sorted_indices             # Ascending (Easy)
            idx_target = torch.flip(sorted_indices, dims=[0]) # Descending (Hard)

            # Re-order data
            xs_target = xs_intensity[idx_target]
            pl_target = pl_t[idx_target]
            conf_target = conf_t[idx_target]
            w_target = hardness[idx_target] # Use HARDNESS here

            # We mix into the "Source" (Easy) batches
            # This aligns with sorting, so we must access xs_intensity via idx_source
            xs_mixed = xs_intensity[idx_source].clone()
            pl_mixed = pl_t[idx_source].clone()
            conf_mixed = conf_t[idx_source].clone()
            w_mixed_map = weight_map[idx_source].clone()

            # Calc Boxes
            v_ratios = [get_adaptive_magnitude(0.2, 0.5, h.item()) for h in hardness]
            bbx1, bby1, bbx2, bby2 = rand_bbox_adaptive((B, C, H, W), v_ratios)

            for i in range(B):
                # Paste Target (Hard) into Source (Easy)
                xs_mixed[i, :, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = xs_target[i, :, bby1[i]:bby2[i], bbx1[i]:bbx2[i]]
                pl_mixed[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = pl_target[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]]
                conf_mixed[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = conf_target[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]]

                # Update weights: The box gets the Target's Hardness
                w_mixed_map[i, :, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = w_target[i]

            # Apply
            xs_cutmix = xs_mixed
            pl_cutmix = pl_mixed
            conf_cutmix = conf_mixed
            weight_map = w_mixed_map

        # 4. Student Forward
        inputs = torch.cat([xw_l, xs_intensity, xs_cutmix], dim=0)

        with amp.autocast(device_type=self.device):
            logits_all, aux_all = self.model(inputs)
            B_l, B_u = xw_l.size(0), xw_u.size(0)

            logits_l = logits_all[:B_l]
            logits_int = logits_all[B_l:B_l+B_u]
            logits_cm = logits_all[B_l+B_u:]

            # 5. Losses
            loss_s = self.loss_fn(logits_l, y_l)
            aux_l = F.interpolate(aux_all[:B_l], size=y_l.shape[-2:], mode='bilinear', align_corners=False)
            loss_s += 0.4 * F.cross_entropy(aux_l, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

            # --- HYBRID FIX: Use Hardness for Weighting ---

            # Unsupervised (Intensity)
            loss_int = F.cross_entropy(logits_int, pl_t, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            mask_int = (conf_t >= conf_tau).float()
            # Weight: Mask * Hardness (Your preferred logic)
            loss_int = (loss_int * mask_int * hardness.view(-1, 1, 1)).sum() / (mask_int.sum() + 1e-6)

            # Unsupervised (CutMix)
            loss_cm = F.cross_entropy(logits_cm, pl_cutmix, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            mask_cm = (conf_cutmix >= conf_tau).float()
            # Weight: Mask * Spatial Hardness Map
            loss_cm = (loss_cm * mask_cm * weight_map.squeeze(1)).sum() / (mask_cm.sum() + 1e-6)

            loss_u = (loss_int + loss_cm) / 2.0
            total_loss = loss_s + lambda_u * loss_u

        return total_loss, hardness.mean()

class IMASTrainer_old(BaseSSLTrainer):
    _METRIC_NAME = "Hardness"

    def train_step(self, batch_l, batch_u):
        # 1. Unpack Inputs
        xw_l, y_l = batch_l['weak'], batch_l['mask']
        xw_u, xs_u = batch_u['weak'], batch_u['strong']

        conf_tau = self.cfg["CONF_TAU"]
        lambda_u = self.cfg["LAMBDA_U"]

        # 2. Teacher Inference (No Grad)
        with torch.no_grad():
            logits_t = self.teacher(xw_u)
            logits_s_weak, _ = self.model(xw_u) # Student (Weak) needed for Hardness

            probs_t = F.softmax(logits_t, dim=1)
            conf_t, pl_t = probs_t.max(1)

            # --- OFFICIAL iMAS HARDNESS CALCULATION ---
            # Using the complex logic (IoU + Confidence)
            hardness = compute_ulb_hardness_official(logits_s_weak, logits_t, thresh=conf_tau)
            gamma = hardness.view(-1, 1, 1, 1)

        # 3. Augmentation Mixing
        # A. Intensity Mix (Weak <-> Strong based on hardness)
        xs_intensity = (gamma * xs_u) + ((1.0 - gamma) * xw_u)

        # B. CutMix (Adaptive box size based on hardness)
        xs_cutmix = xs_intensity.clone()
        pl_cutmix = pl_t.clone()
        conf_cutmix = conf_t.clone()

        if self.cfg.get("USE_CUTMIX", True):
            B, C, H, W = xs_u.shape
            v_ratios = [get_adaptive_magnitude(0.2, 0.5, h.item()) for h in hardness]
            bbx1, bby1, bbx2, bby2 = rand_bbox_adaptive((B, C, H, W), v_ratios)
            rand_idx = torch.randperm(B)
            for i in range(B):
                xs_cutmix[i, :, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = xs_u[rand_idx[i], :, bby1[i]:bby2[i], bbx1[i]:bbx2[i]]
                pl_cutmix[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = pl_t[rand_idx[i], bby1[i]:bby2[i], bbx1[i]:bbx2[i]]
                conf_cutmix[i, bby1[i]:bby2[i], bbx1[i]:bbx2[i]] = conf_t[rand_idx[i], bby1[i]:bby2[i], bbx1[i]:bbx2[i]]

        # 4. Student Forward (Concat Everything)
        # Order: [Labeled, Unlabeled_Intensity, Unlabeled_CutMix]
        inputs = torch.cat([xw_l, xs_intensity, xs_cutmix], dim=0)

        with amp.autocast(device_type=self.device):
            logits_all, aux_all = self.model(inputs)
            B_l, B_u = xw_l.size(0), xw_u.size(0)

            logits_l = logits_all[:B_l]
            logits_int = logits_all[B_l:B_l+B_u]
            logits_cm = logits_all[B_l+B_u:]

            # 5. Losses
            # Supervised
            loss_s = self.loss_fn(logits_l, y_l)
            aux_l = F.interpolate(aux_all[:B_l], size=y_l.shape[-2:], mode='bilinear', align_corners=False)
            loss_s += 0.4 * F.cross_entropy(aux_l, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

            # Unsupervised (Intensity)
            loss_int = F.cross_entropy(logits_int, pl_t, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            mask_int = (conf_t >= conf_tau).float()
            # Mean over pixels first, then weight by hardness
            loss_int = (loss_int * mask_int).sum(dim=(1,2)) / (mask_int.sum(dim=(1,2)) + 1e-6)
            loss_int = (loss_int * hardness).mean()

            # Unsupervised (CutMix)
            loss_cm = F.cross_entropy(logits_cm, pl_cutmix, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            mask_cm = (conf_cutmix >= conf_tau).float()
            # Mean over pixels first, then weight by hardness
            loss_cm = (loss_cm * mask_cm).sum(dim=(1,2)) / (mask_cm.sum(dim=(1,2)) + 1e-6)
            loss_cm = (loss_cm * hardness).mean()

            # Average
            loss_u = (loss_int + loss_cm) / 2.0

            total_loss = loss_s + lambda_u * loss_u

        return total_loss, hardness.mean()

In [ ]:
# ==============================================================================
# --- STRATEGY: U2PL (Faithful Implementation) ---
# ==============================================================================

class ClassWiseQueue:
    def __init__(self, num_classes, queue_len=5000, feature_dim=256, device='cuda'):
        self.num_classes = num_classes
        self.queue_len = queue_len
        self.device = device
        self.bank = torch.randn(num_classes, queue_len, feature_dim, device=device)
        self.ptr = torch.zeros(num_classes, dtype=torch.long, device=device)
        self.full = torch.zeros(num_classes, dtype=torch.bool, device=device)

    @torch.no_grad()
    def enqueue(self, features, labels):
        for c in range(self.num_classes):
            mask = (labels == c)
            if not mask.any(): continue
            feats_c = features[mask]
            n = feats_c.shape[0]
            if n > self.queue_len:
                idx = torch.randperm(n, device=self.device)[:self.queue_len]
                feats_c = feats_c[idx]; n = self.queue_len
            ptr = self.ptr[c].item()
            if ptr + n <= self.queue_len:
                self.bank[c, ptr:ptr+n] = feats_c
                self.ptr[c] = (ptr + n) % self.queue_len
                if self.ptr[c] == 0: self.full[c] = True
            else:
                rem = self.queue_len - ptr
                self.bank[c, ptr:] = feats_c[:rem]
                self.bank[c, :n-rem] = feats_c[rem:]
                self.ptr[c] = n - rem
                self.full[c] = True

    def get_negatives(self, class_ids, num_negatives=256):
        B = class_ids.shape[0]
        negatives = torch.zeros(B, num_negatives, self.bank.shape[-1], device=self.device)
        for i, c in enumerate(class_ids):
            c = c.item()
            if self.full[c] or self.ptr[c] > 0:
                max_idx = self.queue_len if self.full[c] else self.ptr[c]
                idx = torch.randint(0, max_idx, (num_negatives,), device=self.device)
                negatives[i] = self.bank[c, idx]
            else:
                negatives[i] = torch.randn(num_negatives, self.bank.shape[-1], device=self.device)
        return negatives

class U2PLTrainer(BaseSSLTrainer):
    _METRIC_NAME = "Ratio_Reliable"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_classes = HPARAMS["NUM_CLASSES"]

        # Load HPARAMS
        self.queue_len = self.cfg.get("QUEUE_LEN", 30000)
        self.tau = self.cfg.get("TEMP", 0.5)
        self.lambda_c = self.cfg.get("LAMBDA_C", 0.1)
        self.alpha_0 = self.cfg.get("DROP_PERCENT", 30.0)
        self.base_lambda_u = self.cfg.get("LAMBDA_U", 1.0)
        self.use_cutmix = self.cfg.get("USE_CUTMIX", True)

        # Queue & Prototypes
        self.queue = ClassWiseQueue(self.num_classes, queue_len=self.queue_len, feature_dim=256, device=self.device)
        self.register_buffer("prototypes", torch.zeros(self.num_classes, 256, device=self.device))
        self.register_buffer("proto_counts", torch.zeros(self.num_classes, dtype=torch.long, device=self.device))

        # Steps
        steps_per_epoch = len(ULB_FILES) // HPARAMS["BATCH_SIZE"]
        self.total_steps = steps_per_epoch * HPARAMS["NUM_EPOCHS_GEN_0"]
        if self.total_steps == 0: self.total_steps = 1

    def register_buffer(self, name, tensor):
        setattr(self, name, tensor)

    def _update_prototypes(self, features, labels):
        for c in torch.unique(labels):
            # Defensive check against 255 or out of bounds
            if c >= self.num_classes: continue

            mask = (labels == c)
            if not mask.any(): continue
            curr_feat = features[mask].mean(0)
            if self.proto_counts[c] == 0:
                self.prototypes[c] = curr_feat.detach()
            else:
                self.prototypes[c] = 0.9 * self.prototypes[c] + 0.1 * curr_feat.detach()
            self.prototypes[c] = F.normalize(self.prototypes[c], dim=0)
            self.proto_counts[c] += 1

    def apply_cutmix(self, img_u, target_u, valid_u, img_l, target_l):
        B, C, H, W = img_u.shape
        img_mixed = img_u.clone()
        target_mixed = target_u.clone()
        valid_mixed = valid_u.clone()

        lam = np.random.beta(1.0, 1.0)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        perm = torch.randperm(B, device=self.device)
        img_l_shuff = img_l[perm]
        target_l_shuff = target_l[perm]

        img_mixed[:, :, bby1:bby2, bbx1:bbx2] = img_l_shuff[:, :, bby1:bby2, bbx1:bbx2]
        target_mixed[:, bby1:bby2, bbx1:bbx2] = target_l_shuff[:, bby1:bby2, bbx1:bbx2]
        valid_mixed[:, bby1:bby2, bbx1:bbx2] = True

        return img_mixed, target_mixed, valid_mixed

    def train_step(self, batch_l, batch_u):
        img_l, y_l = batch_l['weak'], batch_l['mask']
        img_w, img_s = batch_u['weak'], batch_u['strong']

        # 1. Supervised
        with amp.autocast(device_type=self.device):
            logits_l, aux_l, reps_l = self.model(img_l, return_features=True)
            loss_sup = self.loss_fn(logits_l, y_l)
            if aux_l is not None:
                aux_l = F.interpolate(aux_l, y_l.shape[-2:], mode='bilinear', align_corners=False)
                loss_sup += 0.4 * F.cross_entropy(aux_l, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

        # 2. Teacher
        with torch.no_grad():
            logits_w, _, reps_w = self.teacher(img_w, return_features=True)
            probs_w = F.softmax(logits_w, dim=1)
            entropy = -torch.sum(probs_w * torch.log(probs_w + 1e-10), dim=1)

            progress = min(1.0, self.current_step / max(1, self.total_steps))
            current_drop_pct = self.alpha_0 * (1.0 - progress)
            e_flat = entropy.view(-1)
            k = int(e_flat.numel() * (100 - current_drop_pct) / 100)
            k = max(1, k)
            thresh_val = torch.kthvalue(e_flat, k).values

            mask_reliable = (entropy < thresh_val)
            max_probs, p_labels = probs_w.max(1)

        # 3. Apply CutMix
        if self.use_cutmix:
            img_s, p_labels, mask_reliable = self.apply_cutmix(img_s, p_labels, mask_reliable, img_l, y_l)
            mask_unreliable = ~mask_reliable
        else:
            mask_unreliable = ~mask_reliable

        # 4. Unsupervised
        with amp.autocast(device_type=self.device):
            logits_s, _, reps_s = self.model(img_s, return_features=True)

            # --- CRITICAL FIX: Add ignore_index because p_labels contains 255 from CutMix ---
            loss_u = F.cross_entropy(logits_s, p_labels, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_u = (loss_u * mask_reliable.float()).mean()

            # 5. Contrastive Loss
            total_loss_c = torch.tensor(0.0, device=self.device)

            # Update Prototypes (Teacher) - No CutMix data here
            if mask_reliable.any():
                mask_pure_reliable = (entropy < thresh_val)
                teacher_preds = probs_w.argmax(1)
                self._update_prototypes(reps_w.permute(0,2,3,1).reshape(-1, 256)[mask_pure_reliable.view(-1)],
                                      teacher_preds.view(-1)[mask_pure_reliable.view(-1)])

            # Queue Update
            mask_pure_unreliable = ~(entropy < thresh_val)
            idx_unrel = torch.nonzero(mask_pure_unreliable.view(-1)).squeeze(1)
            if idx_unrel.numel() > 0:
                if idx_unrel.numel() > 2000:
                    perm = torch.randperm(idx_unrel.numel(), device=self.device)[:2000]
                    idx_unrel = idx_unrel[perm]
                self.queue.enqueue(reps_w.permute(0,2,3,1).reshape(-1, 256)[idx_unrel],
                                 probs_w.argmax(1).view(-1)[idx_unrel])

            # Anchors - Contains CutMix data (255)
            indices = torch.nonzero(mask_reliable.view(-1)).squeeze(1)
            if indices.numel() > 0:
                if indices.numel() > HPARAMS["BATCH_SIZE"] * 50:
                    indices = indices[torch.randperm(indices.numel(), device=self.device)[:HPARAMS["BATCH_SIZE"]*50]]

                reps_s_flat = reps_s.permute(0,2,3,1).reshape(-1, 256)
                anchor_feats = reps_s_flat[indices]
                anchor_labels = p_labels.view(-1)[indices]

                loss_c_list = []
                for c in torch.unique(anchor_labels):
                    # --- CRITICAL FIX: Skip Ignore Index in Loop ---
                    if c >= self.num_classes: continue

                    a_mask = (anchor_labels == c)
                    anchors = anchor_feats[a_mask]

                    if self.proto_counts[c] > 0:
                        z_pos = self.prototypes[c].unsqueeze(0)
                        negs = self.queue.get_negatives(torch.tensor([c], device=self.device), num_negatives=256).squeeze(0)

                        l_pos = torch.einsum('nc,kc->nk', anchors, z_pos)
                        l_neg = torch.einsum('nc,kc->nk', anchors, negs)

                        logits_c = torch.cat([l_pos, l_neg], dim=1) / self.tau
                        labels_c = torch.zeros(logits_c.shape[0], dtype=torch.long, device=self.device)
                        loss_c_list.append(F.cross_entropy(logits_c, labels_c))

                if len(loss_c_list) > 0:
                    total_loss_c = torch.stack(loss_c_list).mean()

            # Adaptive Weight
            pct_reliable = mask_reliable.float().mean()
            adaptive_lambda = self.base_lambda_u / (pct_reliable + 1e-6)
            adaptive_lambda = torch.clamp(adaptive_lambda, max=5.0)

            total_loss = loss_sup + adaptive_lambda * loss_u + self.lambda_c * total_loss_c

        return total_loss, pct_reliable

In [ ]:
# ==============================================================================
# --- STRATEGY: UniMatch ---
# ==============================================================================

class UniMatchTrainer(BaseSSLTrainer):
    _METRIC_NAME = "Conf_Rate"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conf_thresh = self.cfg.get("CONF_THRESH", 0.95)
        self.lambda_u = self.cfg.get("LAMBDA_U", 1.0)
        self.use_cutmix = self.cfg.get("USE_CUTMIX", True)

    def apply_cutmix(self, img_u, target_u, valid_u, img_l, target_l):
        B, C, H, W = img_u.shape
        img_mixed = img_u.clone()
        target_mixed = target_u.clone()
        valid_mixed = valid_u.clone()

        lam = np.random.beta(1.0, 1.0)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        perm = torch.randperm(B, device=self.device)
        img_l_shuff = img_l[perm]
        target_l_shuff = target_l[perm]

        img_mixed[:, :, bby1:bby2, bbx1:bbx2] = img_l_shuff[:, :, bby1:bby2, bbx1:bbx2]
        target_mixed[:, bby1:bby2, bbx1:bbx2] = target_l_shuff[:, bby1:bby2, bbx1:bbx2]
        valid_mixed[:, bby1:bby2, bbx1:bbx2] = True

        return img_mixed, target_mixed, valid_mixed

    def train_step(self, batch_l, batch_u):
        img_l, y_l = batch_l['weak'], batch_l['mask']
        img_w = batch_u['weak']
        img_s1, img_s2 = batch_u['strong1'], batch_u['strong2']

        # 1. Supervised Loss
        with amp.autocast(device_type=self.device):
            logits_l, aux_l = self.model(img_l)
            loss_sup = self.loss_fn(logits_l, y_l)
            if aux_l is not None:
                aux_l = F.interpolate(aux_l, y_l.shape[-2:], mode='bilinear', align_corners=False)
                loss_sup += 0.4 * F.cross_entropy(aux_l, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

        # 2. Teacher
        with torch.no_grad():
            logits_w = self.teacher(img_w)
            probs_w = F.softmax(logits_w, dim=1)
            conf_w, p_labels = probs_w.max(1)
            mask_u = (conf_w >= self.conf_thresh)

        # 3. CutMix
        if self.use_cutmix:
            img_s1, target_s1, mask_s1 = self.apply_cutmix(img_s1, p_labels, mask_u, img_l, y_l)
            img_s2, target_s2, mask_s2 = self.apply_cutmix(img_s2, p_labels, mask_u, img_l, y_l)
        else:
            target_s1, mask_s1 = p_labels, mask_u
            target_s2, mask_s2 = p_labels, mask_u

        # 4. Student (Dual Strong)
        img_strong_all = torch.cat([img_s1, img_s2], dim=0)
        with amp.autocast(device_type=self.device):
            logits_strong_all, _ = self.model(img_strong_all)
            logits_s1, logits_s2 = logits_strong_all.chunk(2, dim=0)

            # Loss S1
            loss_s1 = F.cross_entropy(logits_s1, target_s1, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_s1 = (loss_s1 * mask_s1.float()).mean()

            # Loss S2
            loss_s2 = F.cross_entropy(logits_s2, target_s2, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_s2 = (loss_s2 * mask_s2.float()).mean()

        # 5. Student (Feature Perturbation)
        with amp.autocast(device_type=self.device):
            logits_fp = self.model(img_w, feature_perturb=True)
            loss_fp = F.cross_entropy(logits_fp, p_labels, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_fp = (loss_fp * mask_u.float()).mean()

        # 6. Total Unsupervised Loss
        # Paper Eq 11: Loss_FP + 0.5 * (Loss_S1 + Loss_S2)
        loss_u = loss_fp + 0.5 * (loss_s1 + loss_s2)

        total_loss = loss_sup + self.lambda_u * loss_u

        return total_loss, mask_u.float().mean()

In [ ]:
# ==============================================================================
# --- STRATEGY: UniMatch v2 ---
# ==============================================================================


class UniMatchV2Trainer(BaseSSLTrainer):
    _METRIC_NAME = "Conf_Rate"

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conf_thresh = self.cfg.get("CONF_THRESH", 0.95)
        self.lambda_u = self.cfg.get("LAMBDA_U", 1.0)
        self.use_cutmix = self.cfg.get("USE_CUTMIX", True)

    def apply_cutmix(self, img_u, target_u, valid_u, img_l, target_l):
        # Same logic as V1
        B, C, H, W = img_u.shape
        img_mixed = img_u.clone()
        target_mixed = target_u.clone()
        valid_mixed = valid_u.clone()

        lam = np.random.beta(1.0, 1.0)
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        perm = torch.randperm(B, device=self.device)
        img_l_shuff = img_l[perm]
        target_l_shuff = target_l[perm]

        img_mixed[:, :, bby1:bby2, bbx1:bbx2] = img_l_shuff[:, :, bby1:bby2, bbx1:bbx2]
        target_mixed[:, bby1:bby2, bbx1:bbx2] = target_l_shuff[:, bby1:bby2, bbx1:bbx2]
        valid_mixed[:, bby1:bby2, bbx1:bbx2] = True

        return img_mixed, target_mixed, valid_mixed

    def train_step(self, batch_l, batch_u):
        img_l, y_l = batch_l['weak'], batch_l['mask']
        img_w = batch_u['weak']
        img_s1, img_s2 = batch_u['strong1'], batch_u['strong2']

        # 1. Supervised Loss
        with amp.autocast(device_type=self.device):
            logits_l, aux_l = self.model(img_l)
            loss_sup = self.loss_fn(logits_l, y_l)
            if aux_l is not None:
                aux_l = F.interpolate(aux_l, y_l.shape[-2:], mode='bilinear', align_corners=False)
                loss_sup += 0.4 * F.cross_entropy(aux_l, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

        # 2. Teacher (Weak) -> Pseudo Labels
        with torch.no_grad():
            logits_w = self.teacher(img_w)
            probs_w = F.softmax(logits_w, dim=1)
            conf_w, p_labels = probs_w.max(1)
            mask_u = (conf_w >= self.conf_thresh)

        # 3. CutMix
        if self.use_cutmix:
            img_s1, target_s1, mask_s1 = self.apply_cutmix(img_s1, p_labels, mask_u, img_l, y_l)
            img_s2, target_s2, mask_s2 = self.apply_cutmix(img_s2, p_labels, mask_u, img_l, y_l)
        else:
            target_s1, mask_s1 = p_labels, mask_u
            target_s2, mask_s2 = p_labels, mask_u

        # 4. Student Dual Stream
        img_strong_all = torch.cat([img_s1, img_s2], dim=0)

        with amp.autocast(device_type=self.device):
            logits_strong_all = self.model(img_strong_all, complementary_dropout=True)
            logits_s1, logits_s2 = logits_strong_all.chunk(2, dim=0)

            # Loss Stream 1
            loss_s1 = F.cross_entropy(logits_s1, target_s1, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_s1 = (loss_s1 * mask_s1.float()).mean()

            # Loss Stream 2
            loss_s2 = F.cross_entropy(logits_s2, target_s2, reduction='none', ignore_index=HPARAMS["IGNORE_INDEX"])
            loss_s2 = (loss_s2 * mask_s2.float()).mean()

            loss_u = (loss_s1 + loss_s2) / 2

        total_loss = loss_sup + self.lambda_u * loss_u

        return total_loss, mask_u.float().mean()

In [ ]:
# ==============================================================================
# --- STRATEGY: SegKC ---
# ==============================================================================




class SegKCTrainer(BaseSSLTrainer):
    _METRIC_NAME = "Junior_Conf"

    def train_step(self, batch_l, batch_u):
        # Unpack Inputs
        img_l, y_l = batch_l['weak'], batch_l['mask']
        img_u = batch_u['weak']

        T = self.cfg.get("TEMP", 1.0)
        tau = self.cfg["CONF_THRESH"]

        # 1. Forward Pass
        img_all = torch.cat([img_l, img_u], dim=0)

        with amp.autocast(device_type=self.device):
            out_sr, out_jr = self.model(img_all)
            logits_sr_all, aux_sr_all = out_sr
            logits_jr_all, aux_jr_all = out_jr

            B_l = img_l.shape[0]

            sup_sr, sup_jr = logits_sr_all[:B_l], logits_jr_all[:B_l]
            unsup_sr, unsup_jr = logits_sr_all[B_l:], logits_jr_all[B_l:]
            aux_sup_sr, aux_sup_jr = aux_sr_all[:B_l], aux_jr_all[:B_l]

            # --- LOSS 1: Supervised ---
            loss_sup_sr = self.loss_fn(sup_sr, y_l)
            if aux_sup_sr is not None:
                aux_sup_sr = F.interpolate(aux_sup_sr, size=y_l.shape[-2:], mode='bilinear', align_corners=False)
                loss_sup_sr += 0.4 * F.cross_entropy(aux_sup_sr, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

            loss_sup_jr = self.loss_fn(sup_jr, y_l)
            if aux_sup_jr is not None:
                aux_sup_jr = F.interpolate(aux_sup_jr, size=y_l.shape[-2:], mode='bilinear', align_corners=False)
                loss_sup_jr += 0.4 * F.cross_entropy(aux_sup_jr, y_l, ignore_index=HPARAMS["IGNORE_INDEX"])

            loss_sup = 0.5 * (loss_sup_sr + loss_sup_jr)

            # --- LOSS 2: Cross Pseudo Supervision ---
            with torch.no_grad():
                probs_sr = F.softmax(unsup_sr, dim=1)
                max_sr, pl_sr = probs_sr.max(1)
                mask_sr = (max_sr >= tau)

                probs_jr = F.softmax(unsup_jr, dim=1)
                max_jr, pl_jr = probs_jr.max(1)
                mask_jr = (max_jr >= tau)

            loss_con_jr = (F.cross_entropy(unsup_jr, pl_sr, reduction='none', ignore_index=255) * mask_sr.float()).mean()
            loss_con_sr = (F.cross_entropy(unsup_sr, pl_jr, reduction='none', ignore_index=255) * mask_jr.float()).mean()

            loss_con = 0.5 * (loss_con_sr + loss_con_jr)

            # --- LOSS 3: Knowledge Distillation ---
            log_prob_jr = F.log_softmax(unsup_jr / T, dim=1)
            target_prob_sr = F.softmax(unsup_sr / T, dim=1).detach()

            # --- FIX: Use 'mean' reduction for pixel-wise loss --
            loss_kd = F.kl_div(log_prob_jr, target_prob_sr, reduction='none').sum(dim=1).mean() * (T**2)

            # --- TOTAL LOSS ---
            lambda_sup, lambda_con, lambda_kd = self.cfg["LAMBDA_SUP"], self.cfg["LAMBDA_CON"], self.cfg["LAMBDA_KD"]
            total_loss = (lambda_sup * loss_sup) + (lambda_con * loss_con) + (lambda_kd * loss_kd)

        return total_loss, mask_sr.float().mean()

In [ ]:
# ==============================================================================
# --- 4d) TRAINER FACTORY (REGISTRY) ---
# ==============================================================================

def get_trainer(method_name, model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg_override=None):
    method = method_name.upper()
    # 1. Load Base Config
    cfg = HPARAMS["ALGORITHM_CONFIGS"].get(method, {}).copy()

    # 2. Apply Overrides (e.g. GenX EMA)
    if cfg_override:
        cfg.update(cfg_override)

    print(f"🔧 Building Trainer for strategy: {method} | EMA: {cfg.get('EMA_ALPHA', 'N/A')}")

    if method == "SUPERVISED":
        return SupervisedTrainer(model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg)
    elif method == "IMAS":
        return IMASTrainer(model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg)
    elif method == "U2PL":
        return U2PLTrainer(model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg)
    elif method == "UNIMATCH":
        return UniMatchTrainer(model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg)
    elif method == "UNIMATCH_V2":
        return UniMatchV2Trainer(model, teacher, optimizer, scheduler, scaler, loss_fn, device, cfg)
    elif method == "SEGKC":
        return SegKCTrainer(model, None, optimizer, scheduler, scaler, loss_fn, device, cfg)
    else:
        raise ValueError(f"Unknown SSL Method: {method}. Check HPARAMS['SSL_METHOD'] in Cell 1.")

In [ ]:
# ==============================================================================
# --- 5) MAIN GENERATIONAL TRAINING LOOP ---
# ==============================================================================

# --- A. Logging & State Helpers ---

# Helper to generate dynamic filenames based on config
def get_log_path(base_name):
    method = HPARAMS["SSL_METHOD"]
    frac = HPARAMS["DATASET_FRACTION"]
    # E.g., "training_log_UNIMATCH_V2_1_16.csv"
    filename = f"{base_name.replace('.csv', '')}_{method}_{frac}.csv"
    return os.path.join(DRIVE_EXP_DIR, filename)

# Define Paths dynamically
TRAIN_LOG_PATH    = get_log_path("training_log.csv")
BEST_REC_PATH     = get_log_path("best_overall_records.csv")
SLIDING_LOG_PATH  = get_log_path("sliding_window_log.csv")
HOURLY_LOG_PATH   = get_log_path("hourly_status.csv")
STATE_PATH        = os.path.join(DRIVE_EXP_DIR, f"state_{HPARAMS['SSL_METHOD']}_{HPARAMS['DATASET_FRACTION']}.json")

# Global clock tracker
LAST_CLOCK_TIME = time.time()

def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, 'r') as f: data = json.load(f)
        # Ensure schema compatibility
        if "accumulated_training_seconds" not in data: data["accumulated_training_seconds"] = 0.0
        if "last_logged_hour" not in data: data["last_logged_hour"] = 0
        if "global_best_miou" not in data: data["global_best_miou"] = 0.0
        return data
    else:
        return {
            "current_gen": 0,
            "gen_results": {},
            "accumulated_training_seconds": 0.0,
            "last_logged_hour": 0,
            "global_best_miou": 0.0
        }

def save_state(state):
    with open(STATE_PATH, 'w') as f: json.dump(state, f, indent=4)

def log_train_epoch(gen, epoch, loss, metric, s_miou, t_miou, lr):
    """Standard detailed log for every epoch."""
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    headers = ["timestamp", "generation", "epoch", "loss", "metric", "student_miou", "teacher_miou", "lr"]
    row = [ts, gen, epoch, f"{loss:.4f}", f"{metric:.4f}", f"{s_miou:.4f}", f"{t_miou:.4f}", f"{lr:.2e}"]

    file_exists = os.path.exists(TRAIN_LOG_PATH)
    with open(TRAIN_LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists: writer.writerow(headers)
        writer.writerow(row)

def log_global_best(gen, epoch, model_type, miou, per_class_iou):
    """Only logs when we hit a NEW ALL-TIME HIGH."""
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    headers = ["timestamp", "generation", "epoch", "model_type", "miou"] + CITYSCAPES_CLASS_NAMES
    row = [ts, gen, epoch, model_type, f"{miou:.4f}"] + [f"{iou:.4f}" for iou in per_class_iou]

    file_exists = os.path.exists(BEST_REC_PATH)
    with open(BEST_REC_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists: writer.writerow(headers)
        writer.writerow(row)
    print(f"      🏆 NEW GLOBAL BEST! ({model_type}: {miou:.4f})")

def log_sliding_window(gen, model_type, miou, per_class_iou, filename):
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    headers = ["timestamp", "generation", "model_type", "miou", "filename"] + CITYSCAPES_CLASS_NAMES
    row = [ts, gen, model_type, f"{miou:.4f}", filename] + [f"{iou:.4f}" for iou in per_class_iou]

    file_exists = os.path.exists(SLIDING_LOG_PATH)
    with open(SLIDING_LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists: writer.writerow(headers)
        writer.writerow(row)

def check_hourly_milestone(state):
    """
    Checks if a new hour has passed. Modifies state IN-PLACE.
    """
    global LAST_CLOCK_TIME

    # 1. Update Time Accumulator
    now = time.time()
    delta = now - LAST_CLOCK_TIME
    LAST_CLOCK_TIME = now

    # Ensure field exists
    if "accumulated_training_seconds" not in state:
        state["accumulated_training_seconds"] = 0.0

    state["accumulated_training_seconds"] += delta
    total_hours = state["accumulated_training_seconds"] / 3600.0

    # 2. Check if we crossed an integer hour boundary
    current_hour_milestone = int(total_hours)

    if "last_logged_hour" not in state:
        state["last_logged_hour"] = 0

    last_logged = state["last_logged_hour"]

    if current_hour_milestone > last_logged:
        file_exists = os.path.exists(HOURLY_LOG_PATH)

        with open(HOURLY_LOG_PATH, "a", newline="") as f:
            writer = csv.writer(f)
            if not file_exists:
                writer.writerow(["timestamp", "elapsed_hours", "best_overall_miou"])

            # Fill in gaps (e.g. if an epoch took 2 hours)
            for h in range(last_logged + 1, current_hour_milestone + 1):
                ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
                # Default to 0.0 if not set yet
                best_val = state.get("global_best_miou", 0.0)
                writer.writerow([ts, h, f"{best_val:.4f}"])
                print(f"      ⏰ Hourly Log: Hour {h} | Best So Far: {best_val:.4f}")

        state["last_logged_hour"] = current_hour_milestone

    save_state(state)


# --- B. Pseudo-Label Generation (Inconsistency Masks) ---
@torch.no_grad()
def generate_pseudo_labels(teacher_paths, unlabeled_list, save_dir, device="cuda"):
    print(f"🎨 Generating Pseudo-Labels -> {save_dir}")
    os.makedirs(save_dir, exist_ok=True)

    # 1. Standard Normalize
    tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True), v2.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

    # 2. Determine Architecture for Generation
    # If Gen 0 was SegKC (Dual Model), Gen 1+ MUST use the Junior (Small) model.
    # We must force build_model to create a 'UniMatchV2_Model' (Small), not 'SegKC_Model' (Dual).
    # For all other methods (iMAS/ResNet), we use the original method name.

    original_method = HPARAMS["SSL_METHOD"]

    if original_method == "SEGKC":
        print("   👉 Strategy: SegKC detected. Using 'UNIMATCH_V2' (Junior/Small) architecture for generation.")
        temp_method = "UNIMATCH_V2"
    else:
        temp_method = original_method

    teachers = []

    # 3. Build Models & Load Weights
    # We temporarily swap the global config to trick build_model if necessary
    try:
        HPARAMS["SSL_METHOD"] = temp_method

        for p in teacher_paths:
            # Build the inference architecture
            m = build_model(dropout_p=0.0).to(device)

            # Load weights using the smart loader (handles SegKC extraction)
            load_checkpoint_for_inference(m, p, device)

            m.eval()
            teachers.append(m)

    finally:
        # CRITICAL: Always restore the original configuration
        HPARAMS["SSL_METHOD"] = original_method

    # 4. Sliding Window Logic
    erode_k = HPARAMS["IM_ERODE"]
    dilate_k = HPARAMS["IM_DILATE"]
    tile_size = HPARAMS["CROP_SIZE"]
    overlap = HPARAMS["SW_OVERLAP"]

    def _bin_erode(m, k): return 1 - F.max_pool2d(1 - m, kernel_size=k, stride=1, padding=k//2)
    def _bin_dilate(m, k): return F.max_pool2d(m, kernel_size=k, stride=1, padding=k//2)

    def predict_sliding_window(model, img_tensor):
        B, C, H, W = img_tensor.shape
        stride = int(tile_size * (1 - overlap))
        preds = torch.zeros((B, HPARAMS["NUM_CLASSES"], H, W), device=device)
        count = torch.zeros((B, 1, H, W), device=device)
        rows = int(math.ceil((H - tile_size) / stride) + 1)
        cols = int(math.ceil((W - tile_size) / stride) + 1)
        for r in range(rows):
            for c in range(cols):
                y1 = min(r * stride, H - tile_size); x1 = min(c * stride, W - tile_size)
                y2, x2 = y1 + tile_size, x1 + tile_size
                tile = img_tensor[:, :, y1:y2, x1:x2]
                with amp.autocast(device_type=device):
                    out = model(tile)
                    if isinstance(out, tuple): out = out[0]
                preds[:, :, y1:y2, x1:x2] += out
                count[:, :, y1:y2, x1:x2] += 1.0
        return preds / count

    def predict_tta(model, x):
        out = predict_sliding_window(model, x)
        if HPARAMS["IM_TTA"]:
            x_flip = torch.flip(x, [3])
            out_flip = predict_sliding_window(model, x_flip)
            out += torch.flip(out_flip, [3])
            out /= 2.0
        return out

    # 5. Inference Loop
    for path in tqdm(unlabeled_list, desc="Pseudo-Labelling"):
        img = Image.open(path).convert("RGB")
        x = tf(img).unsqueeze(0).to(device)

        preds = [predict_tta(t, x).argmax(1) for t in teachers]
        preds = torch.stack(preds, dim=0)

        consensus = preds[0].float()
        agreement = (preds == preds[0]).all(dim=0).float()
        im_mask = 1.0 - agreement

        if erode_k > 0: im_mask = _bin_erode(im_mask, erode_k)
        if dilate_k > 0: im_mask = _bin_dilate(im_mask, dilate_k)

        final_mask = consensus.clone()
        final_mask[im_mask > 0.5] = HPARAMS["IGNORE_INDEX"]

        out_path = os.path.join(save_dir, os.path.basename(path).replace('_leftImg8bit.png','_gtFine_labelIds.png'))
        Image.fromarray(final_mask.squeeze().cpu().numpy().astype(np.uint8)).save(out_path)

    del teachers; gc.collect(); torch.cuda.empty_cache()



# --- Save Configuration to Drive ---

config_path = os.path.join(DRIVE_EXP_DIR, f"config_{HPARAMS['SSL_METHOD']}_{HPARAMS['DATASET_FRACTION']}.json")
with open(config_path, 'w') as f:
    # Convert sets/arrays to list for JSON serialization if needed
    json.dump(HPARAMS, f, indent=4, default=str)
print(f"💾 Configuration saved to: {config_path}")


# --- C. Main Execution Loop ---

state = load_state()
start_gen = state["current_gen"]
DEVICE = "cuda"

# Resources
with open(labeled_split_file, 'r') as f: LAB_FILES = f.read().splitlines()
with open(unlabeled_split_file, 'r') as f: ULB_FILES = f.read().splitlines()
class_weights = compute_class_weights(labeled_split_file, ID_TO_TRAINID).to(DEVICE)

# --- RESET CLOCK ON STARTUP ---
LAST_CLOCK_TIME = time.time()

for gen in range(start_gen, HPARAMS["NUM_GENERATIONS"]):
    print(f"\n{'='*40}\n🔄 GENERATION {gen}\n{'='*40}")

    # 1. Config & Data
    dl_u = None

    # Store original method to restore later (Critical for SegKC transition)
    original_ssl_method = HPARAMS["SSL_METHOD"]

    # --- DETERMINE BATCH SIZE ---
    if gen == 0:
        current_bs = HPARAMS["BATCH_SIZE_GEN0"]
    else:
        current_bs = HPARAMS["BATCH_SIZE"]

    print(f"📦 Using Batch Size: {current_bs} for Gen {gen}")


    # --- Validation Skipping ---
    if gen == 0:
        val_start = HPARAMS.get("VAL_START_EPOCH_GEN0", 1)
    else:
        val_start = HPARAMS.get("VAL_START_EPOCH_GENX", 1)


    # --- Optimizer & Dataset Selection Logic ---
    if gen == 0:
        # Check Crop Size for V2
        if HPARAMS["SSL_METHOD"] == "UNIMATCH_V2" and HPARAMS["CROP_SIZE"] % 14 != 0:
             print(f"⚠️ Warning: CROP_SIZE {HPARAMS['CROP_SIZE']} not divisible by 14. Adjusting...")

        target_method = HPARAMS["SSL_METHOD"]
        mode = "std" if target_method == "SUPERVISED" else "ssl"

        # --- SELECT AUGMENTATION STRATEGY ---
        if target_method == "IMAS":
            aug_strategy = iMASAug()
        elif target_method == "UNIMATCH" or target_method == "UNIMATCH_V2":
            aug_strategy = StandardColorAug(mode="dual", strength="strong")
        elif target_method == "U2PL":
            aug_strategy = StandardColorAug(mode="single", strength="standard")
        elif target_method == "SEGKC":
            aug_strategy = StandardColorAug(mode="single", strength="standard")
        else:
            aug_strategy = None

        ds_l = CityscapesLabeled(labeled_split_file, ID_TO_TRAINID, mode=mode, ssl_aug_strategy=aug_strategy)
        sampler_l = RCSSampler(ds_l, ID_TO_TRAINID) if HPARAMS["USE_RCS"] else None
        dl_l = DataLoader(ds_l, batch_size=current_bs, sampler=sampler_l, shuffle=(sampler_l is None),
                          num_workers=4, drop_last=True, persistent_workers=True)

        if target_method != "SUPERVISED":
            ds_u = CityscapesUnlabeled(unlabeled_split_file, ssl_aug_strategy=aug_strategy)
            dl_u = DataLoader(ds_u, batch_size=current_bs, shuffle=True, num_workers=4, drop_last=True, persistent_workers=True)

        loss_name = HPARAMS["LOSS_GEN0"]
        phases = [("Train", int(HPARAMS["NUM_EPOCHS_GEN_0"]), "poly")]
    else:
        # --- GEN 1+: SUPERVISED SELF-TRAINING ---
        target_method = "SUPERVISED"
        loss_name = HPARAMS["LOSS_GENX"]

        # Force the architecture to UNIMATCH_V2 (Single Small Model) for Gen 1+
        # This prevents building the Dual SegKC model again
        if original_ssl_method == "SEGKC":
          HPARAMS["SSL_METHOD"] = "UNIMATCH_V2"

        pseudo_dir = os.path.join(PSEUDO_LABEL_DIR, f"gen_{gen}")
        should_gen = True
        if os.path.exists(pseudo_dir):
            if len(os.listdir(pseudo_dir)) >= len(ULB_FILES) * 0.99: should_gen = False

        if should_gen:
            prev_res = state["gen_results"].get(str(gen-1), {})
            if not prev_res: raise ValueError(f"No models found from Gen {gen-1}!")

            # Sort by performance
            all_sorted_paths = sorted(prev_res.items(), key=lambda x: x[1], reverse=True)

            # --- SEGKC Filter: Prefer Student (Junior) models ---
            if original_ssl_method == "SEGKC":
                student_paths = [p for p, sc in all_sorted_paths if "Student" in p]
                if student_paths:
                    print(f"   🕵️ SegKC Detected: Using Best Student (Junior) models for generation.")
                    top_k_paths = student_paths[:HPARAMS["IM_TEACHERS_TOPK"]]
                else:
                    print("   ⚠️ Warning: No Student models found! Fallback to available.")
                    top_k_paths = [p for p, sc in all_sorted_paths[:HPARAMS["IM_TEACHERS_TOPK"]]]
            else:
                top_k_paths = [p for p, sc in all_sorted_paths[:HPARAMS["IM_TEACHERS_TOPK"]]]

            generate_pseudo_labels(top_k_paths, ULB_FILES, pseudo_dir, DEVICE)

        safe_split = os.path.join(split_dir, f"unlabeled_gen{gen}.txt")
        valid_files = [f for f in ULB_FILES if os.path.exists(os.path.join(pseudo_dir, os.path.basename(f).replace('_leftImg8bit.png','_gtFine_labelIds.png')))]
        with open(safe_split, 'w') as f: f.write('\n'.join(valid_files))

        aug_ctrl = AugScheduleController(HPARAMS["AUG_SCHEDULE_GENX"])
        aug_prov = {"get_scale": aug_ctrl.get_scale_range, "get_transforms": aug_ctrl.get_transforms}
        ds_l = CityscapesLabeled(labeled_split_file, ID_TO_TRAINID, mode="std", aug_provider=aug_prov)

        if HPARAMS["USE_STRONG_PSEUDO_AUG"]:
            aug_psu = AugScheduleController(HPARAMS["AUG_SCHEDULE_GENX_PSEUDO"])
            psu_prov = {"get_scale": aug_psu.get_scale_range, "get_transforms": aug_psu.get_transforms,
                        "get_transforms_strong": lambda: aug_psu.get_transforms(True)}
            ds_p = CityscapesPseudoStrong(safe_split, pseudo_dir, psu_prov)
        else:
            ds_p = CityscapesPseudo(safe_split, pseudo_dir, aug_prov)

        dl_l = DataLoader(ConcatDataset([ds_l, ds_p]), batch_size=current_bs, shuffle=True, num_workers=4, drop_last=True, persistent_workers=True)

        total_epochs = HPARAMS["NUM_EPOCHS_GEN_X"]
        warmup_pct = HPARAMS["SNAPSHOT_WARMUP_PCT"]

        n_warmup = int(total_epochs * warmup_pct)
        n_snapshot = total_epochs - n_warmup

        phases = [("Warmup", n_warmup, "poly"), ("Snapshot", n_snapshot, "cosine")]

    # Determine Dropout Rate
    drop_p = HPARAMS["DROPOUT_SCHEDULE"]["GEN0"] if gen == 0 else HPARAMS["DROPOUT_SCHEDULE"]["GENX"]
    print(f"🏗️ Building Model with Dropout p={drop_p}")

    # 2. Model Setup
    val_dl = DataLoader(ValDataset(val_split_file, ID_TO_TRAINID), batch_size=HPARAMS["VAL_BATCH_SIZE"], shuffle=False, num_workers=2)

    # Note: HPARAMS["SSL_METHOD"] might be temporarily patched to UNIMATCH_V2 here if Gen > 0
    # This guarantees we build the correct Single Model for Gen 1+.
    if target_method == "SEGKC" and gen == 0:
        # --- SegKC Special Setup (Gen 0 Only) ---
        model = build_model(dropout_p=drop_p).to(DEVICE) # Returns SegKC_Model
        teacher = model.senior # Senior acts as teacher
    else:
        # --- Standard Teacher-Student Setup (Gen 0 others, or Gen 1+ all) ---
        model = build_model(dropout_p=drop_p).to(DEVICE)
        teacher = build_model(dropout_p=0.0).to(DEVICE)

        # Restore HPARAMS method name (important for saving/logging)
        HPARAMS["SSL_METHOD"] = original_ssl_method

        # Handle Weight Loading here (before freezing)
        if gen > 0:
             prev_res = state.get("gen_results", {}).get(str(gen-1), {})
             if prev_res:
                # Same filtering logic as above for weight loading
                sorted_paths = sorted(prev_res.items(), key=lambda x: x[1], reverse=True)
                if original_ssl_method == "SEGKC":
                     s_paths = [p for p, s in sorted_paths if "Student" in p]
                     best_prev = s_paths[0] if s_paths else sorted_paths[0][0]
                else:
                     best_prev = sorted_paths[0][0]

                print(f"📥 Initializing from Gen {gen-1}: {os.path.basename(best_prev)}")
                load_checkpoint_for_inference(model, best_prev, DEVICE)

        # Clone to teacher
        teacher.load_state_dict(model.state_dict())

        # Freeze Teacher
        for p in teacher.parameters(): p.requires_grad = False

    # Optimizer Init
    lr_mult = HPARAMS["LR_FINETUNE_FACTOR"] if gen > 0 else 1.0

    if HPARAMS["SSL_METHOD"] == "UNIMATCH_V2" and gen == 0:
        print("⚡ Using AdamW for UniMatch V2")
        lr_enc = 5e-6
        lr_dec = 2e-4
        if gen > 0:
            lr_enc *= lr_mult
            lr_dec *= lr_mult
        params = [
            {'params': [p for n, p in model.named_parameters() if 'backbone' in n], 'lr': lr_enc},
            {'params': [p for n, p in model.named_parameters() if 'backbone' not in n], 'lr': lr_dec},
        ]
        optimizer = torch.optim.AdamW(params, weight_decay=0.01, betas=(0.9, 0.999))
    else:
        optimizer = get_optimizer(model, lr_factor=lr_mult)

    scaler = amp.GradScaler(enabled=True)
    loss_fn = make_loss_fn(loss_name, class_weights)

    trainer_cfg_override = {}
    if gen > 0:
        trainer_cfg_override["EMA_ALPHA"] = HPARAMS.get("EMA_ALPHA_GENX", 0.99)
        trainer_cfg_override.update(HPARAMS["SUP_AUG"])

    trainer = get_trainer(target_method, model, teacher, optimizer, None, scaler, loss_fn, DEVICE, cfg_override=trainer_cfg_override)

    best_s, best_t = 0.0, 0.0
    best_s_path, best_t_path = None, None
    epoch_counter = 1
    state.setdefault("gen_results", {})[str(gen)] = {}

    # 3. Phase Loop
    for phase_name, n_epochs, sched_type in phases:
        if n_epochs <= 0: continue

        steps_per_epoch = len(dl_u) if dl_u else len(dl_l)

        if sched_type == "poly":
            sched = WarmupPolyLR(optimizer, max_iters=steps_per_epoch * n_epochs,
                                 warmup_iters=int(steps_per_epoch * n_epochs * 0.05))
        elif sched_type == "cosine":
            for pg in optimizer.param_groups: pg['lr'] = pg.get('base_lr', pg['lr']); pg['initial_lr'] = pg['lr']
            n_snaps = HPARAMS.get("SNAPSHOT_MODELS_GENX", 5)
            steps_per_cycle = int((steps_per_epoch * n_epochs) // n_snaps)
            sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=steps_per_cycle, T_mult=1, eta_min=1e-6)

        trainer.scheduler = sched

        for _ in range(n_epochs):
            loss, metric = trainer.train_epoch(dl_l, dl_u)
            curr_lr = optimizer.param_groups[0]['lr']

            if epoch_counter < val_start:
                print(f"   [Ep {epoch_counter}] Loss: {loss:.4f} | {trainer.metric_name}: {metric:.3f} | LR: {curr_lr:.2e} (Val Skipped)")
                log_train_epoch(gen, epoch_counter, loss, metric, 0.0, 0.0, curr_lr)
            else:

                use_sliding = (HPARAMS.get("VALIDATION_MODE", "standard") == "sliding")

                # 1. Validate Student
                if use_sliding:
                    miou_s, pci_s = validate_sliding(val_dl, model, DEVICE)
                else:
                    miou_s, _, pci_s = validate_miou_and_ce(val_dl, model, DEVICE)

                # 2. Validate Teacher (Conditional)
                skip_teacher = HPARAMS.get("SKIP_TEACHER_VAL_SUP", False)
                is_supervised_phase = (target_method == "SUPERVISED")

                if is_supervised_phase and skip_teacher:
                    miou_t = 0.0
                    pci_t = [0.0] * HPARAMS["NUM_CLASSES"]
                else:
                    if use_sliding:
                        miou_t, pci_t = validate_sliding(val_dl, teacher, DEVICE)
                    else:
                        miou_t, _, pci_t = validate_miou_and_ce(val_dl, teacher, DEVICE)

                print(f"   [Ep {epoch_counter}] Loss: {loss:.4f} | {trainer.metric_name}: {metric:.3f} | S-mIoU: {miou_s:.4f} | T-mIoU: {miou_t:.4f} | LR: {curr_lr:.2e}")
                log_train_epoch(gen, epoch_counter, loss, metric, miou_s, miou_t, curr_lr)

                # --- 1. Update State with Max ---
                current_epoch_max = max(miou_s, miou_t)
                global_best = state["global_best_miou"]

                # --- 2. Check for Global Best ---
                if current_epoch_max > global_best:
                    state["global_best_miou"] = current_epoch_max
                    # Identify who won for the log
                    if miou_s > global_best:
                        log_global_best(gen, epoch_counter, "Student", miou_s, pci_s)
                    if miou_t > global_best: # Use if because both might beat the old record
                        log_global_best(gen, epoch_counter, "Teacher", miou_t, pci_t)

                # --- 3. Save Gen Best Models ---
                if miou_s > best_s:
                    best_s = miou_s
                    name = f"Gen{gen}_Best_Student_{miou_s:.4f}.pth"
                    best_s_path = os.path.join(MODEL_SAVE_DIR, name)
                    torch.save(model.state_dict(), best_s_path)
                    state["gen_results"][str(gen)][best_s_path] = miou_s

                if miou_t > best_t:
                    best_t = miou_t
                    name = f"Gen{gen}_Best_Teacher_{miou_t:.4f}.pth"
                    best_t_path = os.path.join(MODEL_SAVE_DIR, name)
                    torch.save(teacher.state_dict(), best_t_path)
                    state["gen_results"][str(gen)][best_t_path] = miou_t

            # --- 4. Check Hourly Milestone ---
            # Updates state["accumulated_training_seconds"] and writes CSV if hour passed
            check_hourly_milestone(state)

            if sched_type == "cosine":
                steps_taken = int((_ + 1) * steps_per_epoch)
                cycle_len = int(steps_per_cycle)
                epoch_len = int(steps_per_epoch)

                if (steps_taken % cycle_len) < epoch_len:
                    print(f"      📸 Snapshot Saved! (Teacher: {miou_t:.4f})")
                    name = f"Gen{gen}_Snapshot_Teacher_{miou_t:.4f}.pth"
                    path = os.path.join(MODEL_SAVE_DIR, name)
                    torch.save(teacher.state_dict(), path)
                    state["gen_results"][str(gen)][path] = miou_t
                    if epoch_counter >= val_start and miou_t > state["global_best_miou"]:
                         log_global_best(gen, epoch_counter, "Snapshot", miou_t, pci_t)

            save_state(state)
            epoch_counter += 1

    # 4. End of Gen
    print(f"\n📏 End of Gen {gen}: Running Sliding Window Validation on Bests...")
    if best_s_path:
        print(f"   -> Testing Best Student: {os.path.basename(best_s_path)}")
        model.load_state_dict(torch.load(best_s_path))
        sw_miou, sw_pci = validate_sliding(val_dl, model, DEVICE)
        log_sliding_window(gen, "Student", sw_miou, sw_pci, os.path.basename(best_s_path))

    if best_t_path:
        print(f"   -> Testing Best Teacher: {os.path.basename(best_t_path)}")

        if target_method == "SEGKC":
            # --- FIX: SegKC Specific Loading ---
            # The teacher checkpoint only contains the Senior submodule weights.
            # We load them into 'teacher' (which is model.senior), NOT 'model'.
            teacher.load_state_dict(torch.load(best_t_path))

            # We validate 'teacher' directly.
            # This works because teacher is a UniMatchV2_Model (standard backbone+decoder)
            sw_miou, sw_pci = validate_sliding(val_dl, teacher, DEVICE)
        else:
            # --- Standard SSL Loading ---
            # Load teacher weights into the main model architecture for validation
            model.load_state_dict(torch.load(best_t_path))
            sw_miou, sw_pci = validate_sliding(val_dl, model, DEVICE)

        log_sliding_window(gen, "Teacher", sw_miou, sw_pci, os.path.basename(best_t_path))

    state["current_gen"] = gen + 1
    save_state(state)

print("🏁 Experiment Complete.")